# 計算機程式期中專案：小恐龍 & 寶石方塊 全自動玩家

| 項目 | 內容 |
|---|---|
| 學號 | C115143151 |
| 姓名 | 金鼎富 |
| 課程 | 計算機程式 |
| 程式語言 | Python 3（Jupyter Notebook） |

## 專案摘要

兩個遊戲都用同一個架構：**看（截圖）→ 想（判斷）→ 做（按鍵盤、動滑鼠）**，一直重複（小恐龍每秒看幾百張畫面）。

| | 小恐龍 | 寶石方塊（Facebook 版 Bejeweled Blitz） |
|---|---|---|
| 怎麼看 | 截取恐龍前方一長條，找出障礙物在哪裡 | 截取整個盤面，用 Lab 色彩空間把每格分成第幾種寶石 |
| 怎麼想 | **量障礙物速度 → 算出還有幾秒會撞到**，再用遊戲物理推算最佳起跳時機，並**自動量測電腦的反應延遲** | 把所有可能的交換（8×8 約 112 種）都模擬一次，連「消掉 → 掉落 → 再消」的連鎖也一起算，挑分數最高的 |
| 怎麼做 | 空白鍵跳、按↓蹲下；跳過障礙物後**立刻按↓快速落地** | 拖曳或點擊交換寶石；**寶石還在掉落時，就在已經靜止的地方繼續下**（加速模式） |
| 自動適應 | 自動找恐龍與地面、畫面放大（chrome://dino 街機模式約 3 倍）、夜晚模式反色、網頁捲動、電腦卡頓；避開 Chrome 的「慢速模式」陷阱 | 自動判斷交換方式、遊戲是否允許「掉落中繼續下」；認出並使用超立方體；等盤面出現才下、1 分鐘到或被結束畫面蓋住就停 |
| 開發測試結果 | chrome://dino（新版街機模式）實測：連續 9 分鐘沒撞到（10,220 分，測試時間到才停），全程最高速 | 自製 Blitz 型測試版（1 分鐘、可邊掉邊下）：遊戲收到 108～120 次交換，14.4～15.7 萬分，約是「等盤面停下來」模式的 1.8 倍 |

> **遵守規則**：只使用老師提供的網址；程式只模擬一般玩家的「看螢幕＋鍵盤＋滑鼠」，**沒有修改遊戲、沒有讀取遊戲內部資料、沒有課金或使用道具**。
> 寶石方塊的 Boost、稀有寶石（Rare Gem）、加時（Encore）都要用金幣，而金幣可以花錢買，所以**全部不用**；程式在 1 分鐘到之前就停手，也會在結束畫面出現時立刻停，不會點到加時按鈕。

## 使用步驟（Windows）

1. 安裝 Python（[python.org](https://www.python.org/downloads/)，安裝時勾選 **Add python.exe to PATH**）。
2. 開啟這個筆記本，兩種方法都可以：
   - **VS Code**：直接打開這個檔案。第一次要選 Python：選「Global」→「Python 3.x」；跳出要安裝 `ipykernel` 就按「安裝」。
   - **瀏覽器版 Jupyter**：雙擊和這個檔案放在一起的 **`啟動Jupyter.bat`**（會自動安裝套件並打開 Jupyter），再點開這個筆記本。
3. 第一次先按上方的「**全部執行**」（Run All），然後**手離開滑鼠**，等最下面的第 C 部分出現 ✅。
   需要操作遊戲的格子（A-3、A-7、B-5、B-6）會自動跳過：**倒數時滑鼠沒動、或滑鼠還停在 VS Code／Jupyter 上，程式就不會點任何東西、不按任何鍵**。
4. **小恐龍**：用 Chrome **開一個新分頁**打開老師給的小恐龍網址（例如 `chrome://dino`）→ 執行 A-7 → 5 秒內切到遊戲視窗，把滑鼠放在恐龍上，不要動，**也不要自己點恐龍**（程式會自己點恐龍右邊的空白處再開始，原因見 A-2）。
5. **寶石方塊**：用 Chrome 打開老師給的網址（Facebook 版 Bejeweled Blitz）並登入 → **開一局練習**，在遊戲進行中執行 B-5（校正＋檢查辨識）→ 正式比賽：執行 B-6，10 秒內切到遊戲、確認沒有選任何 Boost、按下開始，程式會等盤面出現才下，玩滿 1 分鐘自動停。
   （重新打開 VS Code／Jupyter 之後，要先再按一次「全部執行」，把所有函式載入。）
6. 緊急停止：把滑鼠**快速甩到螢幕左上角**，或按 Jupyter 工具列的 ■。

> ⚠️ 必須在**自己的電腦**上執行（Google Colab 在雲端，看不到也控制不了你的螢幕）。

## 目錄
- 第 0 部分：環境設定
- 第 A 部分：小恐龍（A-1 參數 → A-8 賽後分析）
- 第 B 部分：寶石方塊（B-1 參數 → B-7 賽後分析）
- 第 C 部分：自我測試（不用開遊戲）
- 開發過程紀錄、參考資料

---
# 第 0 部分：環境設定

| 套件 | 用途 |
|---|---|
| `pyautogui` | 控制鍵盤和滑鼠（按空白鍵、拖曳寶石） |
| `mss` | 超快速截圖（每次只要 1～3 毫秒，小恐龍需要每秒看幾百次） |
| `numpy` | 把截圖變成數字陣列來計算（找障礙物、算顏色、模擬盤面） |
| `matplotlib` | 畫出校正結果、辨識結果、賽後分析圖 |

第一次使用要安裝（之後不用再執行安裝那格）。

In [ ]:
%pip install pyautogui mss numpy matplotlib

下面這格是兩個遊戲共用的工具。重點：
- **Windows 螢幕縮放修正**：筆電常設成 125%／150% 縮放，不處理的話截圖座標和滑鼠座標會對不上。
- **`pyautogui.PAUSE = 0`**：pyautogui 預設每個動作後停 0.1 秒，對小恐龍來說太慢。
- **`grab()`**：截圖並統一成 RGB；Mac Retina 螢幕截圖會是 2 倍大，這裡自動縮回來。

In [ ]:
# ==========================================================
# 共用工具：截圖、灰階、倒數、按鍵
# ==========================================================
import sys
import time
import platform

# ---- Windows 螢幕縮放（125%、150%）修正 ----
# 筆電常把「顯示縮放」設成 125% 或 150%，如果不處理，截圖座標和滑鼠座標會對不起來。
# 這兩行讓 Python 用「真實像素」計算，必須在 import pyautogui / mss 之前執行。
if platform.system() == "Windows":
    import ctypes
    try:
        ctypes.windll.shcore.SetProcessDpiAwareness(2)
    except Exception:
        try:
            ctypes.windll.user32.SetProcessDPIAware()
        except Exception:
            pass

import numpy as np     # 數字陣列運算（處理圖片）
import pyautogui       # 控制鍵盤滑鼠
import mss             # 快速截圖

pyautogui.PAUSE = 0        # pyautogui 預設每個動作後停 0.1 秒，對小恐龍太慢 → 改成 0
pyautogui.FAILSAFE = True  # 安全機制：滑鼠甩到螢幕左上角 → 程式立刻停止

if sys.platform == "win32":
    # Windows 截圖加速：mss 預設會連「半透明的浮動視窗」一起截（CAPTUREBLT），
    # 在很多電腦上會讓每張截圖慢到 30～40 毫秒（每秒只能看 25 張，小恐龍根本來不及跳）。
    # 關掉之後通常快 5～10 倍，遊戲畫面一樣截得到。
    import importlib
    for _name in ("mss.windows.gdi", "mss.windows"):       # 新版 mss（10.2 以後）放在 gdi，舊版直接在 windows
        try:
            _m = importlib.import_module(_name)
            if hasattr(_m, "CAPTUREBLT"):
                _m.CAPTUREBLT = 0
        except ImportError:
            pass
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning, module="mss")
warnings.filterwarnings("ignore", message=".*mss.mss is deprecated.*")
_sct = mss.mss()           # 截圖工具（建立一次，重複使用，比較快）

# matplotlib 畫圖時要能顯示中文（Windows 用微軟正黑體，Mac 用蘋方）
try:
    import matplotlib.pyplot as plt
    plt.rcParams["font.sans-serif"] = ["Microsoft JhengHei", "Microsoft YaHei", "PingFang TC", "Heiti TC",
                                       "Noto Sans CJK TC", "Arial Unicode MS", "DejaVu Sans"]
    plt.rcParams["axes.unicode_minus"] = False
except ImportError:
    print("提醒：還沒安裝 matplotlib，畫圖功能不能用（自動玩不受影響）")


def grab(left, top, width, height):
    """
    截取螢幕上的一塊長方形，回傳 numpy 陣列，形狀 = (高, 寬, 3)，顏色順序 RGB。
    left, top：左上角座標；width, height：寬、高（單位：像素）
    """
    region = {"left": int(left), "top": int(top),
              "width": max(1, int(width)), "height": max(1, int(height))}
    img = np.asarray(_sct.grab(region))[:, :, 2::-1]       # mss 給 BGRA → 轉成 RGB
    ratio = img.shape[1] // region["width"]                # Mac Retina 螢幕截圖會是 2 倍大
    if ratio > 1:
        img = img[::ratio, ::ratio]                        # 縮回來，讓座標和滑鼠一致
    return img


def to_gray(img):
    """彩色 → 灰階（0 = 黑，255 = 白）。很多判斷只需要看「深淺」就夠了。"""
    return img.mean(axis=2)


def window_title_at(x, y):
    """（Windows 專用）螢幕上 (x, y) 這一點是哪個視窗？回傳視窗標題；查不到就回傳空字串"""
    if platform.system() != "Windows":
        return ""
    try:
        import ctypes
        from ctypes import wintypes
        user32 = ctypes.WinDLL("user32")              # 自己開一份，不影響 pyautogui
        user32.WindowFromPoint.argtypes = [wintypes.POINT]
        user32.WindowFromPoint.restype = wintypes.HWND
        user32.GetAncestor.argtypes = [wintypes.HWND, wintypes.UINT]
        user32.GetAncestor.restype = wintypes.HWND
        hwnd = user32.GetAncestor(user32.WindowFromPoint(wintypes.POINT(int(x), int(y))), 2)   # 2 = 最外層視窗
        buf = ctypes.create_unicode_buffer(512)
        user32.GetWindowTextW(hwnd, buf, 512)
        return buf.value
    except Exception:
        return ""


def countdown(msg, sec=5):
    """
    倒數計時，讓你有時間把滑鼠移到遊戲上；回傳倒數結束時的滑鼠座標。
    安全機制（回傳 None，呼叫的程式就不會點任何東西、不按任何鍵）：
      ‧ 倒數期間滑鼠完全沒動 → 通常是按了「全部執行」，沒有人在操作
      ‧ 倒數結束時滑鼠還停在 VS Code／Jupyter 上 → 忘了切到遊戲視窗
    """
    print(msg)
    start = pyautogui.position()
    moved = False
    for i in range(sec, 0, -1):
        print(f"  {i} ...", end="", flush=True)
        for _ in range(10):
            time.sleep(0.1)
            moved = moved or pyautogui.position() != start
    print()
    pos = pyautogui.position()
    if not moved:
        print("⚠️ 倒數時滑鼠完全沒動 → 為了安全，這次不點任何東西、不按任何鍵。"
              "（按「全部執行」時看到這行是正常的；要真的開始，就重新執行這一格，倒數時把滑鼠移過去）")
        return None
    title = window_title_at(*pos)
    if "Visual Studio Code" in title or "Jupyter" in title:
        print(f"⚠️ 滑鼠還停在「{title}」上 → 不點任何東西。請重新執行，倒數時切到遊戲視窗、把滑鼠移過去")
        return None
    return pos


print("工具載入完成！螢幕大小：", pyautogui.size(), "／作業系統：", platform.system())

def capture_speed(sec=0.5):
    """測截圖速度：每秒能截幾張 900×120 的圖（小恐龍需要每秒 100 張以上才穩）"""
    n, t0 = 0, time.perf_counter()
    while time.perf_counter() - t0 < sec:
        grab(0, 0, 900, 120)
        n += 1
    return n / sec


fps = capture_speed()
print(f"截圖速度：每秒 {fps:.0f} 張", "✅" if fps >= 100 else "⚠️ 偏慢：關掉其他程式、Chrome 只開遊戲分頁再試")

---
# 第 A 部分：小恐龍

## 原理

```
         ┌──────────── 截圖範圍（每秒截幾百張）────────────────┐
         │  恐龍    │空出│      往前看 6 倍身高                  │
         │  ▟█▙     │0.5 │   [ 蹲下偵測帶 ] ← 頭部高度的飛鳥       │
         │ ███▛     │身寬│   [ 跳躍偵測帶 ] ← 仙人掌、貼地飛鳥     │
  ───────┴──腳──────┴────┴──────────────────────────────────────┴── 地面線
           ↑ 看腳有沒有離地（用來量延遲、判斷落地）
```

1. **自動找恐龍**：按下開始後，在畫面上找「又長又細的地面線」和「站在線上的恐龍」，量出恐龍的身高 H 和寬度，之後所有距離都用「幾倍身高」換算，所以不管螢幕大小、瀏覽器縮放都能用。`chrome://dino` 開始後會進入「街機模式」：畫面放大約 3 倍、恐龍移到左下角，所以先在滑鼠附近找、找不到就搜整個螢幕，而且要「連續三次量到一樣」才算數。
   開始前要先點一下讓 Chrome 收到鍵盤，但**不能點恐龍**：新版 Chrome 有無障礙設計，「先點恐龍再按鍵」會自動打開「Start slower（慢速模式）」，速度和跳躍物理全變、分數漲很慢。所以程式點的是**恐龍右邊的空白網頁**，而且會先確認那裡跟恐龍在同一個視窗。
2. **量速度**：記錄障礙物位置「每次改變」的時間（螢幕一秒更新 60 次，但程式一秒截幾百張，同一格畫面會被截到好幾次，只記第一次），用最近 0.4 秒的點做**最小平方法直線擬合**，斜率就是速度——電腦忙、畫面一下快一下慢的時候，比只看頭尾兩點準很多。而且**遊戲速度只會變快不會變慢**，所以用「最近 3 個障礙物量到的速度」當下限，避免低估（低估速度 = 以為還很遠 = 跳太晚）。
3. **算還有幾秒撞到**：距離 ÷ 速度。遊戲越跑越快，用「時間」決定起跳，就不用猜遊戲多快變快。
   障礙物進到嘴巴前面那一小段「緩衝區」（為了不把自己的頭當成障礙物，這段不看）之後，就用「最後看清楚的位置 − 速度 × 經過時間」**推算**它在哪裡；畫面卡住一下、或夜晚模式切換那一瞬間整片變灰看不清楚，也一樣繼續推算，不會跟丟。
4. **起跳時機（物理推算）**：從原版遊戲原始碼查到的跳躍物理，算出「按下跳躍後，腳要多久才會抬到最高仙人掌上方」（約 0.095 秒）。再考慮恐龍的腳在嘴巴後面一段距離（真正撞到的是腳）。
5. **自動量延遲**：每次按下跳躍後，看恐龍的腳什麼時候離地，就知道「這台電腦從按鍵到畫面反應」要多久，起跳時機會自動提前這麼多。換電腦也不用重調。延遲忽快忽慢時，取最近 9 次裡偏慢的那一邊（第 70 百分位數）；速度越快再多提早一點（`MARGIN_HI`），因為高速時恐龍在空中的時間很充裕，早跳一點沒關係、晚跳就會撞。
6. **快速落地**：跳過障礙物之後，預測障礙物尾巴什麼時候通過恐龍尾巴，那一刻按↓讓恐龍快速落下，馬上準備跳下一個。**這是高速時能活下來的關鍵**（不快速落地的話，最高速時間隔最短的仙人掌會來不及跳）。
7. **夜晚模式**：700 分後畫面會反色（白變黑），而且有 1.5 秒的漸變。背景色每次重新算，判斷門檻跟著「地面線的對比」一起變小變大，漸變中也看得到。
8. **捲動防護**：有些網站按空白鍵或↓會讓網頁捲動，畫面跑掉。程式發現「背景沒變但地面線不見了」時，會重新找「地面線 + 恐龍」，把截圖範圍移過去。
9. **撞到後不亂按**：障礙物 0.5 秒完全不動 = 遊戲結束，程式立刻放開所有按鍵、不再按（不然會把 GAME OVER 畫面按成重新開始，老師就看不到分數了），畫面靜止 2.5 秒後自動停止。

## 從原版遊戲原始碼查到的物理數據（Chromium t-rex-runner）

| 項目 | 數值 | 程式怎麼用 |
|---|---|---|
| 起始速度 / 最高速度 | 每格畫面 6 → 13 像素（每秒 60 格） | `START_SPEED`：還沒量到速度前的預設值 |
| 加速度 | 每格畫面 +0.001 | 約 2 分鐘到最高速 → 一定要會「量」速度 |
| 恐龍身高、寬度 | 47、44 像素（蹲下時寬 59） | 偵測區要離嘴巴 0.5 倍身寬，不然蹲下時會看到自己 |
| 起跳初速、重力 | −10 −速度/10、每格 +0.6 | 推算 `JUMP_LEAD`：腳抬到 50 像素（最高仙人掌）約 6 格畫面 |
| 按↓快速落地 | 下降速度 ×3 | `FAST_DROP` |
| 飛鳥高度 | 離地 0～40／25～65／50～90 像素 | 低的跳、中的蹲、高的不理（亂跳反而會撞到） |
| 障礙物最小間距 | 障礙物寬 × 速度 + 120×0.6 | 最高速時最近只有 0.5 秒 → 需要快速落地 |
| 新版 chrome://dino「慢速模式」（Start slower） | 速度 4.2 → 最高 9、重力 0.25、起跳初速 −20 | 只有「先點恐龍再按鍵」才會打開 → 程式改點恐龍旁邊；萬一已經是慢速模式，程式也照樣能玩（測試 5 分鐘沒撞到） |
| 新版街機模式（chrome://dino） | 畫布放大到視窗寬度（1920 寬螢幕約 3.2 倍） | 開始後才找恐龍、找不到就搜整個螢幕 |

## A-1 參數

**最常調的是 `MARGIN`**：常常「跳太晚撞到」就調大 0.01～0.02；常常「跳太早落在仙人掌上」就調小。

In [ ]:
# ==========================================================
# A-1 參數（單位：H = 恐龍身高；秒）
# ==========================================================
LOOK      = 6.0          # 往前看多遠（6 倍身高）：要夠遠，才來得及量出障礙物的速度
GAP       = 0.5          # 偵測區從嘴巴往前「0.5 倍恐龍寬度」才開始（蹲下的恐龍比站著長 0.34 倍，不能把自己當障礙物）
LOW_BAND  = (0.48, 0.25) # 跳躍偵測帶：地面往上 0.25H～0.48H（仙人掌、貼地飛鳥一定會經過）
MID_BAND  = (0.85, 0.60) # 蹲下偵測帶：地面往上 0.60H～0.85H（頭部高度的飛鳥會經過）
MIN_DIFF  = 10           # 像素跟背景至少差這麼多才算「有東西」
DIFF_RATIO = 0.4         # 門檻 = 地面線對比 × 0.4（夜晚模式漸變時，門檻會自動跟著變）
GROUP_GAP = 0.30         # 兩個障礙物距離小於 0.3H，視為同一組（連在一起的仙人掌）

START_SPEED = 9.0        # 一開始的速度（還沒量到之前先用）：每秒約 9 個身高（6 像素/格 × 60 格/秒 ÷ 約 40 像素身高）
JUMP_LEAD = 0.095        # 按下跳躍後，腳抬到最高仙人掌上方要多久（秒，由遊戲物理推算）
LEG_OFFSET = 0.45        # 恐龍的腳在嘴巴後面「0.45 倍恐龍寬度」（撞到的是腳，不是嘴巴）
MARGIN    = 0.025        # 安全餘裕（秒）：常常太晚跳就調大，太早跳就調小
MARGIN_HI = 0.05         # 高速時再多提早的時間（秒）：速度 11→20 身高/秒，從 0 慢慢加到 0.05
                         # （高速時恐龍在空中的時間很充裕，早跳一點沒關係；電腦偶爾卡一下時，晚跳就會撞到）
LAG_GUESS = 0.06         # 還沒量到延遲之前，先假設「按鍵 → 看到恐龍起跳」要 0.06 秒
LIFT_DETECT = 0.025      # 遊戲開始起跳 → 程式看出腳離地，大約慢 1.5 格畫面
JUMP_HOLD = 0.13         # 空白鍵按住 0.13 秒 → 跳到最高（跳過最寬的仙人掌群）
FAST_DROP = True         # 跳過障礙物後立刻按↓，快速落地（高速時的關鍵）
DROP_MARGIN = 0.0        # 障礙物尾巴過了恐龍尾巴後，再等幾秒才按↓（落在仙人掌上就調大）
DUCK_LEAD = 0.15         # 飛鳥：提早多久蹲下（秒）。蹲早一點沒關係，蹲太晚會被撞
MAX_TIME  = 3600         # 最多玩幾秒就自動停

## A-2 自動找恐龍（函式定義）

In [ ]:
# ==========================================================
# A-2 自動找恐龍：只要把滑鼠放在恐龍附近，程式自己量出位置和大小
# ==========================================================

def longest_run(row):
    """一列 True/False 裡，最長的「連續 True」有多長（用來找又長又直的地面線）"""
    d = np.diff(np.concatenate(([0], row.astype(np.int8), [0])))
    starts, ends = np.flatnonzero(d == 1), np.flatnonzero(d == -1)
    return int((ends - starts).max()) if starts.size else 0


def flood(mask, sy, sx):
    """
    從 (sy, sx) 這一點出發，找出所有「連在一起」的 True 像素（上下左右斜角都算相連）。
    回傳這一團像素的外框：(上, 下, 左, 右)
    """
    h, w = mask.shape
    seen = np.zeros_like(mask, dtype=bool)
    stack = [(sy, sx)]
    seen[sy, sx] = True
    top, bottom, left, right = sy, sy, sx, sx
    while stack:
        y, x = stack.pop()
        top, bottom = min(top, y), max(bottom, y)
        left, right = min(left, x), max(right, x)
        for dy in (-1, 0, 1):
            for dx in (-1, 0, 1):
                ny, nx = y + dy, x + dx
                if 0 <= ny < h and 0 <= nx < w and mask[ny, nx] and not seen[ny, nx]:
                    seen[ny, nx] = True
                    stack.append((ny, nx))
    return top, bottom, left, right


def ground_rows(ink, min_len):
    """
    找「地面線」候選：一條又長又細的水平線。
    條件：這一列連續很長，但它上面第 3 列幾乎是空的（排除整片黑色的區域，例如工作列、視窗外面）。
    """
    runs = np.array([longest_run(row) for row in ink])
    rows = []
    for r in range(3, len(runs) - 5):
        if runs[r] >= min_len and runs[r - 3] < 0.3 * runs[r] and runs[r + 5] < 0.6 * runs[r]:
            if not rows or r - rows[-1] > 3:          # 同一條線的相鄰幾列只取最上面那列
                rows.append(r)
    return rows


def dino_on(ink, gr):
    """
    在地面線（第 gr 列）上面找恐龍：腳踩在地面線上、「最左邊」那一團夠大的像素。
    為什麼是最左邊？恐龍永遠在畫面左邊，仙人掌和飛鳥都是從右邊跑過來的。
    回傳 (上, 下, 左, 右) 或 None
    """
    above = ink[:gr]
    seen = np.zeros_like(above)
    for x in np.flatnonzero(above[gr - 1]):           # 緊貼地面線那一列，從左到右每個點
        if seen[gr - 1, x]:
            continue
        t, b, l, r = flood(above, gr - 1, x)
        seen[t:b + 1, l:r + 1] = True
        H, W = gr - t, r - l + 1
        if H >= 15 and 0.6 < W / H < 1.5:             # 大小和形狀像恐龍（身寬約等於身高）
            return t, b, l, r
    return None


def find_in(left, top, w, h, hint_y):
    """在螢幕上 (left, top, w, h) 這塊範圍裡找「地面線 + 站在上面的恐龍」，找不到回傳 None"""
    img = grab(left, top, w, h)
    gray = to_gray(img)
    bg = np.median(gray)                      # 背景色 = 出現最多的顏色（中位數）
    ink = np.abs(gray - bg) > 40              # 跟背景差很多的像素 = 「有東西」
    for gr in sorted(ground_rows(ink, max(150, 0.2 * min(w, 1000))), key=lambda r: abs(top + r - hint_y)):
        box = dino_on(ink, gr)
        if box:
            return img, gray, ink, bg, gr, box
    return None


def locate_dino(hint_x, hint_y, box_w=900, box_h=520):
    """
    自動找出：
      1. 地面線：又長又細的水平線
      2. 恐龍：站在地面線上、最左邊那一團夠大的深色像素
    先在滑鼠附近 box_w × box_h 的範圍找；找不到就搜整個螢幕
    （chrome://dino 按下開始後會進入「街機模式」：恐龍放大約 3 倍、移到畫面左下角，常常跑出滑鼠附近的範圍）。
    有好幾條候選線時，選「上面真的站著恐龍」、而且離滑鼠最近的那一條。
    回傳一個字典 CAL，裡面是之後所有計算要用的位置。
    """
    sw, sh = pyautogui.size()
    left = int(min(max(0, hint_x - box_w / 2), max(0, sw - box_w)))
    top = int(min(max(0, hint_y - box_h / 2), max(0, sh - box_h)))
    found = find_in(left, top, min(box_w, sw - left), min(box_h, sh - top), hint_y)
    if found is None:                         # 滑鼠附近找不到 → 搜整個螢幕
        left, top = 0, 0
        found = find_in(0, 0, sw, sh, hint_y)
    if found is None:
        raise RuntimeError("找不到恐龍或地面線：請確認遊戲已經開始、整個遊戲畫面都在螢幕上")
    img, gray, ink, bg, ground_r, (t, b, l, r) = found
    H = ground_r - t                           # 恐龍身高（頭頂到地面線）

    line_ink = np.median(gray[ground_r][ink[ground_r]])
    CAL = {
        "H": int(H),                           # 恐龍身高（像素）→ 之後所有距離都用「幾倍身高」換算
        "GROUND_Y": int(top + ground_r),       # 地面線在螢幕上的 y 座標
        "NOSE_X": int(left + r),               # 恐龍嘴巴（最右邊）的 x 座標
        "DINO_LEFT": int(left + l),            # 恐龍尾巴（最左邊）的 x 座標
        "DINO_TOP": int(top + t),              # 恐龍頭頂的 y 座標
        "BG": float(bg),                       # 背景顏色（判斷網頁有沒有被捲動用）
        "CONTRAST": float(abs(line_ink - bg)), # 地面線和背景的深淺差
        "SNAPSHOT": (img, left, top),          # 校正時的截圖（畫圖檢查用）
    }
    return CAL


def focus_point(hint_x, hint_y):
    """
    開始前要先「點一下」，Chrome 才會收到鍵盤。但是【不能點在恐龍身上】！
    原因：新版 Chrome 小恐龍有「無障礙模式」——如果先點恐龍（遊戲區被選取）再按鍵開始，
    它會以為你在用鍵盤導覽，自動打開「Start slower（慢速模式）」：
    速度變慢（最高 9，正常是 13）、跳躍的物理全部改變，分數漲得慢很多。
    所以改點「恐龍右邊的空白網頁」：一樣能讓 Chrome 收到鍵盤，又不會進入慢速模式。
    （還沒開始時，遊戲區只有恐龍那一小塊；開始之後才變大，那時點到就沒關係了）
    安全檢查：要點的位置必須跟滑鼠在【同一個視窗】，不然寧可不點旁邊。
    回傳 (x, y)，找不到安全的點就回傳 None
    """
    sw, sh = pyautogui.size()
    R = 160                                               # 在滑鼠附近 320×320 的範圍找恐龍
    left, top = int(max(0, hint_x - R)), int(max(0, hint_y - R))
    w, h = int(min(sw - left, 2 * R)), int(min(sh - top, 2 * R))
    gray = to_gray(grab(left, top, w, h))
    ink = np.abs(gray - np.median(gray)) > 40             # 跟背景差很多 = 有東西
    ys, xs = np.nonzero(ink)
    cands = []
    if xs.size:
        hx, hy = hint_x - left, hint_y - top
        i = int(np.argmin((xs - hx) ** 2 + (ys - hy) ** 2))   # 離滑鼠最近的「有東西」的點
        if (xs[i] - hx) ** 2 + (ys[i] - hy) ** 2 < 80 ** 2:
            t, b, l, r = flood(ink, int(ys[i]), int(xs[i]))    # 那一整團 = 恐龍
            size = max(b - t, r - l, 20)
            cands += [(left + r + max(1.5 * size, 100), top + (t + b) / 2),   # 恐龍右邊的空白處
                      (left + (l + r) / 2, top + b + 1.5 * size)]     # 恐龍下面的空白處
    cands.append((hint_x + 200, hint_y))                  # 找不到恐龍：往右邊 200 像素
    title = window_title_at(hint_x, hint_y)
    for x, y in cands:
        x, y = int(min(max(x, 0), sw - 1)), int(min(max(y, 0), sh - 1))
        if window_title_at(x, y) == title:                # 同一個視窗才點
            return x, y
    return None


def start_and_locate(hint_x, hint_y, min_wait=1.2, max_wait=4.0):
    """
    點一下遊戲旁邊 → 按空白鍵開始 → 等畫面穩定 → 自動找恐龍。
    為什麼開始之後才找？很多網站按下空白鍵後，畫面會展開、放大（chrome://dino 的「街機模式」會放大約 3 倍），
    網頁也可能往下捲，恐龍的位置和大小都會變。
    所以每 0.25 秒量一次，連續三次量到一樣才算數。（開始 3 秒後才會出現第一個障礙物，時間很夠）
    就算之後畫面又變了，玩的時候也會自動重新找恐龍（見 A-6）。
    """
    fp = focus_point(hint_x, hint_y)
    if fp is None:
        fp = (hint_x, hint_y)
        print("⚠️ 恐龍旁邊找不到可以安全點的空白處，只好直接點恐龍。"
              "如果畫面出現「Start slower」勾選框（慢速模式），請按 F5 重新整理後再跑一次。")
    pyautogui.click(*fp)                       # 點一下，讓瀏覽器專心在遊戲上（點旁邊，不點恐龍）
    time.sleep(0.15)
    if "chrome://dino" in window_title_at(*fp).lower():
        # 慢速模式一旦打開，這個分頁就會一直維持慢速，只有重新整理才會恢復。
        # 所以每次開始前都先按 F5 重新整理（只在 chrome://dino 的視窗才按），保證是一般速度。
        pyautogui.press("f5")
        time.sleep(2.0)                        # 等網頁重新載入完（重新整理後鍵盤還是在網頁上，不用再點）
    pyautogui.press("space")                   # 開始遊戲（在「遊戲結束」畫面時，這也會重新開始）
    t0 = time.time()
    time.sleep(min_wait)
    prev, same = None, 0
    while time.time() - t0 < max_wait:
        try:
            cal = locate_dino(hint_x, hint_y)
        except RuntimeError:
            cal = None
        if cal and prev and all(abs(cal[k] - prev[k]) <= 1 for k in ("H", "GROUND_Y", "NOSE_X", "DINO_LEFT")):
            same += 1
            if same >= 2:
                return cal                     # 連續三次一樣 → 畫面穩定了
        else:
            same = 0
        prev = cal
        time.sleep(0.25)
    if prev:
        print("⚠️ 畫面一直在變，先用最後一次量到的位置")
        return prev
    raise RuntimeError("找不到恐龍：請確認遊戲畫面完整出現在螢幕上，滑鼠放在恐龍附近")


def show_calibration(CAL):
    """把校正結果畫出來：紅框 = 恐龍，綠線 = 地面，黃色 = 跳躍偵測帶，藍色 = 蹲下偵測帶"""
    import matplotlib.pyplot as plt
    import matplotlib.patches as patches
    img, left, top = CAL["SNAPSHOT"]
    H = CAL["H"]
    fig, ax = plt.subplots(figsize=(7, 7))
    ax.imshow(img)
    ax.add_patch(patches.Rectangle((CAL["DINO_LEFT"] - left, CAL["DINO_TOP"] - top),
                                   CAL["NOSE_X"] - CAL["DINO_LEFT"], CAL["GROUND_Y"] - CAL["DINO_TOP"],
                                   fill=False, color="red", lw=2, label="恐龍"))
    ax.axhline(CAL["GROUND_Y"] - top, color="green", lw=1, label="地面線")
    x0 = CAL["NOSE_X"] + GAP * (CAL["NOSE_X"] - CAL["DINO_LEFT"]) - left
    for (a, b), c, name in ((LOW_BAND, "gold", "跳躍偵測帶"), (MID_BAND, "deepskyblue", "蹲下偵測帶")):
        ax.add_patch(patches.Rectangle((x0, CAL["GROUND_Y"] - a * H - top), LOOK * H, (a - b) * H,
                                       color=c, alpha=0.35, label=name))
    ax.legend(loc="upper right")
    ax.set_title(f"恐龍身高 H = {H} 像素")
    ax.axis("off")
    plt.show()

## A-3 校正測試（選用）

想先確認程式有沒有找對恐龍時執行。執行後 5 秒內切到遊戲視窗、滑鼠放在恐龍上，程式會**自動開始遊戲**並畫出找到的位置（恐龍沒人控制會撞死，沒關係）。
倒數時沒移動滑鼠（例如按了「全部執行」），這格會自動跳過，不會點任何東西。
紅框要框住恐龍、綠線要在地面上、黃色和藍色的偵測帶要在恐龍前方。

In [ ]:
# 校正測試（不會開始自動玩）：倒數時切到遊戲視窗，把滑鼠放在【恐龍身上】
pos = countdown("把滑鼠移到【小恐龍身上】，停住不要動", sec=5)
if pos:                                       # 倒數時沒移動滑鼠（例如按了「全部執行」）就跳過
    CAL = start_and_locate(*pos)
    print(f"找到恐龍！身高 H = {CAL['H']} 像素，地面 y = {CAL['GROUND_Y']}，"
          f"嘴巴 x = {CAL['NOSE_X']}，尾巴 x = {CAL['DINO_LEFT']}")
    show_calibration(CAL)

## A-4 恐龍的「眼睛」

一次截圖同時看三件事：**跳躍偵測帶**、**蹲下偵測帶**、**恐龍的腳**。連在一起（間隔 < 0.3H）的仙人掌當成同一組，記下這組的「頭」和「尾」。

In [ ]:
# ==========================================================
# A-4 恐龍的「眼睛」：每次截一張圖，回報前方有沒有障礙物
# ==========================================================
def first_group(cols, max_gap):
    """
    cols：每一直行有沒有障礙物（True/False）
    回傳最靠近恐龍的那「一組」障礙物的 (開始, 結束) 位置；中間空隙 ≤ max_gap 算同一組
    """
    idx = np.flatnonzero(cols)
    if idx.size == 0:
        return None
    gaps = np.flatnonzero(np.diff(idx) > max_gap)       # 哪裡開始出現大空隙
    end = idx[gaps[0]] if gaps.size else idx[-1]
    return int(idx[0]), int(end)


class DinoEye:
    """
    截圖範圍：從恐龍尾巴開始，往前到「嘴巴 + 0.5 倍身寬 + 6H」，高度從地面往上 0.95H，往下 3 像素。
    一次截圖同時看到：恐龍的腳（判斷有沒有離地）＋ 前方的兩條偵測帶。
    如果地面線突然不見（但背景顏色沒變），代表畫面被捲動或放大縮小 → self.lost = True，交給主迴圈重新找恐龍。
    """
    def __init__(self, cal):
        H = self.H = cal["H"]
        self.x0 = cal["DINO_LEFT"]
        self.y0 = int(cal["GROUND_Y"] - 0.95 * H)
        self.dw = cal["NOSE_X"] - cal["DINO_LEFT"]                 # 恐龍寬度
        self.view = int(self.dw * (1 + GAP))                        # 偵測區從截圖的第幾行開始
        self.w = int(self.view + LOOK * H)
        self.h = int(cal["GROUND_Y"] + 3 - self.y0)
        self.gr = cal["GROUND_Y"] - self.y0                         # 地面線在截圖裡的第幾列

        def rows(a, b):                                             # 「地面往上 a～b 倍身高」→ 第幾列
            return slice(int(self.gr - a * H), int(self.gr - b * H))
        self.low_rows, self.mid_rows = rows(*LOW_BAND), rows(*MID_BAND)
        self.leg_rows = rows(0.30, 0.08)                            # 恐龍腳的位置（判斷起跳/落地）
        self.leg_cols = slice(int(0.15 * self.dw), int(0.65 * self.dw))
        self.sky_rows = slice(0, max(1, self.gr - 2))
        self.max_gap = GROUP_GAP * H
        self.bg0, self.contrast0 = cal["BG"], cal["CONTRAST"]
        self.lost = False                                           # 地面線不見了（畫面被捲動或放大縮小）

    def look(self):
        """截一張圖，回傳 (時間, 下帶障礙物, 上帶障礙物, 腳的像素數, 灰階圖, 門檻)"""
        t = time.perf_counter()
        gray = to_gray(grab(self.x0, self.y0, self.w, self.h))
        bg = np.median(gray[self.sky_rows, self.view:])             # 背景色（白天白、晚上黑都適用）
        # 地面線的對比：夜晚模式切換時，整個畫面會慢慢變成灰色，對比變小 → 門檻也跟著變小
        contrast = max(abs(np.median(gray[r, self.view:]) - bg)
                       for r in (self.gr - 1, self.gr, self.gr + 1))
        # 背景顏色沒變（不是夜晚模式），地面線卻不見了 → 畫面被捲動或放大縮小了
        self.lost = contrast < 0.25 * self.contrast0 and abs(bg - self.bg0) < 25
        thr = max(MIN_DIFF, DIFF_RATIO * contrast)
        mask = np.abs(gray - bg) > thr                              # True = 有東西
        low = mask[self.low_rows, self.view:].sum(axis=0) >= 2      # 每一直行至少 2 點才算
        mid = mask[self.mid_rows, self.view:].sum(axis=0) >= 2
        legs = int(mask[self.leg_rows, self.leg_cols].sum())
        return t, first_group(low, self.max_gap), first_group(mid, self.max_gap), legs, gray, thr

## A-5 恐龍的「大腦」

狀態機（同一時間只會是其中一種狀態）：

```
RUN（地上跑）──前方有障礙物、剩餘時間 ≤ 起跳時機──▶ AIR（跳躍中）──障礙物尾巴過了恐龍尾巴──▶ DROP（按↓快速落地）──落地──▶ RUN
   └──只有頭部高度有飛鳥──▶ DUCK（蹲下）──飛鳥過去了──▶ RUN
任何狀態 ──障礙物 0.5 秒完全不動（撞到了）──▶ STUCK（放開按鍵，不再按）
```

為什麼「落地」要等障礙物尾巴過去才算？仙人掌從恐龍腳下經過時，「腳的位置」會出現仙人掌的像素，看起來像落地了，其實恐龍還在空中。

所有按鍵都是「排程」：按下空白鍵後**不 sleep**，而是記下「什麼時候放開」，讓眼睛可以繼續看（不然按住的 0.13 秒內程式是瞎的）。

In [ ]:
# ==========================================================
# A-5 恐龍的「大腦」：量速度 → 算還有幾秒撞到 → 決定動作
# ==========================================================
class SpeedMeter:
    """
    量障礙物的速度（像素/秒）：
      1. 記錄「最近障礙物的位置」每次改變的時間（同一格畫面被截到好幾次，只記第一次）
      2. 用最近 0.4 秒的點做「最小平方法直線擬合」：斜率 = 速度
         （比只看頭尾兩點更不怕畫面卡頓：電腦忙的時候，遊戲畫面有時 15 毫秒更新一次、有時 60 毫秒）
      3. 遊戲速度只會越來越快、不會變慢 → 用「最近幾個障礙物量到的速度」當下限。
         低估速度 = 以為還很遠 = 跳太晚 = 死，所以寧可估高一點點。
    """
    def __init__(self, v0):
        self.v = v0
        self.hist = []                       # [(第一次看到這個位置的時間, 位置), ...]
        self.none_since = None               # 從什麼時候開始看不到障礙物
        self.track_v = None                  # 這一個障礙物量到的可靠速度
        self.recent = []                     # 最近幾個障礙物各自量到的速度
        self.floor = 0.0                     # 速度下限

    def end_track(self):
        """一個障礙物看完了：把它的速度存起來，更新下限（取最近 3 個的中位數，不怕單一次量錯）"""
        if self.track_v:
            self.recent = (self.recent + [self.track_v])[-5:]
            if len(self.recent) >= 3:
                self.floor = 0.93 * float(np.median(self.recent[-3:]))   # 飛鳥可能比地面慢 7%，留一點空間
        self.track_v = None
        self.hist.clear()

    def scale(self, k):
        """畫面放大縮小 k 倍：速度也跟著變 k 倍"""
        self.v *= k
        self.floor *= k
        self.recent = [r * k for r in self.recent]
        self.end_track()

    def update(self, x, t):
        if x is None:
            self.none_since = self.none_since or t
            if t - self.none_since > 0.15:   # 真的沒東西了（不是只閃一下）→ 這個障礙物結束
                self.end_track()
            return
        self.none_since = None
        if x <= 1:
            return                           # 障礙物已經進到「緩衝區」，位置被卡在 0 → 不準，不拿來算速度
        if self.hist and x > self.hist[-1][1] + 3:
            self.end_track()                 # 位置突然變遠 = 換成下一個障礙物了
        if not self.hist or x != self.hist[-1][1]:
            self.hist.append((t, x))
        while self.hist and t - self.hist[0][0] > 0.4:
            self.hist.pop(0)                 # 只用最近 0.4 秒
        if len(self.hist) >= 4:
            ts = np.array([h[0] for h in self.hist])
            xs = np.array([h[1] for h in self.hist], dtype=float)
            span = ts[-1] - ts[0]
            if span >= 0.05:
                v = -np.polyfit(ts - ts[0], xs, 1)[0]      # 直線擬合的斜率（往左跑，所以加負號）
                if v > 0:
                    if span >= 0.12 and len(self.hist) >= 6:
                        self.v = 0.6 * self.v + 0.4 * v      # 資料夠多 → 直接相信
                        self.track_v = v
                    elif 0.5 * self.v < v < 2.0 * self.v:    # 資料少時，太離譜的數值不採用
                        self.v = 0.8 * self.v + 0.2 * v
        self.v = max(self.v, self.floor)


class EdgeTracker:
    """
    推算「障礙物最前面」現在在哪裡（像素，0 = 偵測區的起點），解決三個問題：
      1. 緩衝區：障礙物進到嘴巴前面那一小段（為了不把自己的頭當成障礙物，這段不看）之後，
         眼睛只會回報「位置 = 0」→ 用「最後看清楚的位置 − 速度 × 經過的時間」推算。
         速度慢的時候（剛開始、或慢速模式）特別重要：該起跳的那一刻，障礙物已經在緩衝區裡了。
      2. 畫面卡頓：同一格畫面可能被截到好幾次，但障礙物其實一直在往前跑
         → 從「第一次看到這個位置」開始，用時間推算它現在大概走到哪。
      3. 短暫看不見（例如切換夜晚模式那一瞬間，整個畫面變灰）→ 0.25 秒內繼續推算，不會跟丟。
    """
    BLIND = 0.25

    def __init__(self):
        self.anchor = None                   # (看清楚的位置, 第一次看到的時間)
        self.width = 0                       # 這組障礙物多寬
        self.none_since = None

    def update(self, group, t, v):
        if group is None:                    # 現在看不到
            if self.anchor is None:
                return None
            self.none_since = self.none_since or t
            if t - self.none_since > self.BLIND:
                self.anchor = None           # 太久看不到 → 真的沒有了
                return None
            ax, at = self.anchor
            return ax - v * (t - at)         # 短暫看不見：繼續推算
        self.none_since = None
        x = group[0]
        if x > 1:                            # 看得清楚
            if self.anchor is None or x != self.anchor[0]:
                self.anchor = (x, t)         # 新位置：記下第一次看到的時間
            self.width = group[1] - group[0]
            ax, at = self.anchor
            return ax - v * (t - at)
        if self.anchor is None:              # 一出現就在緩衝區裡（很少見）→ 當作就在 0
            return x
        ax, at = self.anchor
        return min(x, ax - v * (t - at))     # 在緩衝區裡：用速度推算（可以是負的 = 更靠近了）


class DinoBot:
    """
    狀態：RUN（地上跑）→ AIR（跳起來）→ DROP（按↓快速落地）→ RUN
                       ↘ DUCK（蹲下躲鳥）→ RUN
          障礙物 0.5 秒完全不動（撞到了、遊戲結束）→ STUCK：放開按鍵、不再按任何鍵
    所有按鍵都用「排程」：按下去之後不 sleep，讓眼睛可以繼續看。
    """
    def __init__(self, cal):
        self.H, self.dw = cal["H"], cal["NOSE_X"] - cal["DINO_LEFT"]
        self.speed = SpeedMeter(START_SPEED * self.H)
        self.state = "RUN"
        self.lags = []                 # 量到的延遲：按下跳 → 看到腳離地（秒）
        self.legs_base = None          # 恐龍在地上時，腳的像素數（基準）
        self.release_at = None         # 什麼時候放開空白鍵
        self.log = []                  # 紀錄：(時間, 事件, 剩幾秒撞到, 速度 H/秒, 延遲)
        self.low_edge, self.mid_edge = EdgeTracker(), EdgeTracker()   # 推算障礙物現在在哪

    def rescale(self, cal):
        """畫面放大縮小後：用新的恐龍大小重新換算（速度單位是像素/秒，也要跟著放大縮小）"""
        k = cal["H"] / self.H
        self.H, self.dw = cal["H"], cal["NOSE_X"] - cal["DINO_LEFT"]
        self.speed.scale(k)
        self.legs_base = None
        self.low_edge, self.mid_edge = EdgeTracker(), EdgeTracker()
        for key in ("space", "down"):
            pyautogui.keyUp(key)
        self.state, self.release_at = "RUN", None

    def lag(self):
        """
        目前估計的延遲：取最近 9 次裡「偏慢的那一邊」（第 70 百分位數）。
        電腦忙的時候延遲忽快忽慢（0.05～0.15 秒都有），用偏慢的估計 → 寧可早一點跳。
        （高速時恐龍在空中的時間很充裕，早跳 0.03 秒沒關係；晚跳 0.03 秒就會撞到）
        """
        return float(np.percentile(self.lags[-9:], 70)) if self.lags else LAG_GUESS

    def go_stuck(self, t):
        """遊戲結束了：放開按鍵，之後不再按任何鍵"""
        for key in ("space", "down"):
            pyautogui.keyUp(key)
        self.release_at = None
        self.state, self.stuck_t = "STUCK", t
        self.log.append((t, "stuck", 0.0, self.speed.v / self.H, self.lag()))

    def t_jump(self, v):
        """還剩幾秒會撞到時要起跳（物理推算 + 自動量到的延遲）"""
        game_lag = max(0.0, self.lag() - LIFT_DETECT)
        extra = MARGIN_HI * min(1.0, max(0.0, (v / self.H - 11) / 9))     # 越快越提早
        return max(0.02, game_lag + JUMP_LEAD + MARGIN + extra - LEG_OFFSET * self.dw / v)

    def step(self, t, low, mid, legs):
        """看完一張圖之後呼叫：決定要按什麼鍵"""
        H = self.H
        if (low, mid) != getattr(self, "seen", None):        # 障礙物畫面有沒有在動（判斷遊戲是不是停了）
            self.seen, self.last_change = (low, mid), t
        self.relaxed = False                                 # True = 附近沒東西，主迴圈可以稍微休息

        # 遊戲凍結偵測：看得到障礙物，它卻 0.5 秒完全沒動 → 遊戲結束（撞到了）
        # → 放開所有按鍵、進入 STUCK，之後不再按任何鍵（不然會把「遊戲結束」畫面按成重新開始）
        frozen = (low or mid) and t - self.last_change > 0.5
        if self.state == "STUCK":
            if t - self.stuck_t > 1.5 and t - self.last_change < 0.1:
                self.state = "RUN"                           # 畫面又動了（例如手動重新開始）→ 繼續玩
                self.low_edge, self.mid_edge = EdgeTracker(), EdgeTracker()
            return
        if frozen:
            self.go_stuck(t)
            return
        near = min([g[0] for g in (low, mid) if g], default=None)
        self.speed.update(near, t)
        v = self.speed.v
        gap_px = GAP * self.dw                                   # 偵測區和嘴巴之間的距離

        # 腳的基準：在地上跑的時候慢慢更新
        if self.state == "RUN":
            self.legs_base = legs if self.legs_base is None else 0.95 * self.legs_base + 0.05 * legs
        airborne = self.legs_base is not None and legs < 0.35 * self.legs_base

        # 放開空白鍵（按住時間到了）
        if self.release_at is not None and t >= self.release_at:
            pyautogui.keyUp("space")
            self.release_at = None

        if self.state in ("RUN", "DUCK"):
            low_x = self.low_edge.update(low, t, v)              # 要跳的障礙物最前面在哪（含推算）
            mid_x = self.mid_edge.update(mid, t, v)              # 要蹲的飛鳥最前面在哪
            if low_x is not None:                                # 前面有要跳的東西
                ttc = (low_x + gap_px) / v                       # 還有幾秒撞到嘴巴
                if ttc <= self.t_jump(v):
                    if self.state == "DUCK":
                        pyautogui.keyUp("down")                  # 蹲著不能跳 → 先站起來
                    pyautogui.keyDown("space")
                    self.release_at = t + JUMP_HOLD
                    self.state, self.t_press, self.lifted = "AIR", t, False
                    tail = low[1] if low else low_x + self.low_edge.width
                    self.trail = (tail, t)                       # 記住這組障礙物的尾巴
                    self.low_edge = EdgeTracker()                # 這組處理掉了，之後重新追蹤下一組
                    self.log.append((t, "jump", ttc, v / H, self.lag()))
                    return
            if mid and not low and self.state == "RUN":          # 頭部高度的飛鳥
                ttc = (mid_x + gap_px) / v
                if ttc <= max(0.0, self.lag() - LIFT_DETECT) + DUCK_LEAD + MARGIN:
                    pyautogui.keyDown("down")
                    self.state = "DUCK"
                    self.duck_until = t + (mid[1] + gap_px + self.dw) / v + 0.05
                    self.log.append((t, "duck", ttc, v / H, self.lag()))
                    return
            if self.state == "DUCK" and t >= self.duck_until and not mid:
                pyautogui.keyUp("down")                          # 鳥飛過了 → 站起來
                self.state = "RUN"
            soon = [(x + gap_px) / v for x in (low_x, mid_x) if x is not None]
            self.relaxed = self.state == "RUN" and min(soon, default=9) > 0.4   # 0.4 秒內不會撞到任何東西

        elif self.state == "AIR":
            if low and low[0] <= self.trail[0] + 3:
                self.trail = (low[1], t)                         # 還看得到同一組 → 更新尾巴位置
            if not self.lifted and airborne:
                self.lifted = True                               # 第一次看到腳離地 → 量延遲
                if 0.015 < t - self.t_press < 0.3:               # 不合理的數值不採用
                    self.lags.append(t - self.t_press)
            # 預測：障礙物尾巴什麼時候通過恐龍尾巴
            t_pass = self.trail[1] + (self.trail[0] + gap_px + self.dw) / v
            if FAST_DROP and self.lifted and t >= t_pass + DROP_MARGIN:
                if self.release_at is not None:
                    pyautogui.keyUp("space")
                    self.release_at = None
                pyautogui.keyDown("down")                        # 快速落地
                self.state, self.drop_t = "DROP", t
            elif self.lifted and not airborne and t - self.t_press > 0.2 and t >= t_pass:
                # 自然落地。為什麼要等障礙物過去（t_pass）才算？
                # 仙人掌從恐龍腳下經過時，「腳的位置」會出現仙人掌的像素，看起來像落地了，其實還在空中
                self.state = "RUN"
            elif not self.lifted and t - self.t_press > 0.35:
                if t - self.last_change > 0.25:
                    self.go_stuck(t)                             # 按了跳，恐龍沒動、障礙物也不動了 → 遊戲結束
                else:
                    self.state = "RUN"                           # 畫面還在動：這次按鍵沒反應到，回去重新判斷
            elif t - self.t_press > 1.2:
                self.state = "RUN"                               # 保險：太久沒落地就重設

        elif self.state == "DROP":
            if (not airborne and t - self.drop_t > 0.03) or t - self.drop_t > 0.4:
                pyautogui.keyUp("down")                          # 落地了 → 放開↓
                self.state = "RUN"

## A-6 主迴圈（函式定義）

In [ ]:
# ==========================================================
# A-6 開始玩：眼睛看 → 大腦想 → 手按鍵，一直重複
# ==========================================================
from collections import deque

last_frames = deque(maxlen=60)   # 死前最後 60 張截圖（死掉後拿來分析）
LAST_BOT = None                  # 上一局的大腦（保留紀錄給分析用）


def play_dino(cal):
    """用校正好的 CAL 開始自動玩（遊戲要已經在跑）"""
    global LAST_BOT
    eye, bot = DinoEye(cal), DinoBot(cal)
    LAST_BOT = bot
    last_frames.clear()
    H = cal["H"]
    relocations, last_try, lost_since = 0, 0.0, None
    import gc
    gc.disable()     # 玩的時候暫停 Python 自動整理記憶體（垃圾回收），避免偶爾卡住 0.1 秒來不及跳
    t_start = time.perf_counter()
    still_since, prev, frames = t_start, None, 0
    try:
        while time.perf_counter() - t_start < MAX_TIME:
            t, low, mid, legs, gray, thr = eye.look()     # 1. 看
            if eye.lost:                                  # 畫面變了 → 在原來的位置附近重新找恐龍
                lost_since = lost_since or t
                if t - lost_since > 5:
                    print("恐龍不見了超過 5 秒（遊戲視窗被切走或關掉了？），自動停止")
                    break
                if t - last_try > 0.25:
                    last_try = t
                    try:
                        new = locate_dino((cal["DINO_LEFT"] + cal["NOSE_X"]) / 2, cal["DINO_TOP"] + cal["H"] / 2)
                        cal = new
                        eye = DinoEye(cal)
                        bot.rescale(cal)
                        H = cal["H"]
                        relocations += 1
                    except RuntimeError:
                        pass                              # 還沒找到（畫面可能還在變），等一下再試
                continue
            lost_since = None
            bot.step(t, low, mid, legs)                   # 2. 想 + 3. 按
            frames += 1
            if bot.relaxed:
                time.sleep(0.002)                         # 附近沒有障礙物 → 休息 2 毫秒，把電腦讓給瀏覽器（畫面更順）
            last_frames.append((t - t_start, gray))
            # 遊戲結束偵測：畫面 2.5 秒完全不動
            if prev is not None and np.array_equal(gray, prev):
                if t - still_since > 2.5:
                    print("偵測到畫面靜止，遊戲結束")
                    break
            else:
                still_since = t
            prev = gray
    except KeyboardInterrupt:
        print("手動停止")
    except pyautogui.FailSafeException:
        print("滑鼠移到螢幕左上角 → 緊急停止")
    finally:
        gc.enable()                                       # 恢復自動整理記憶體
        pyautogui.FAILSAFE = False                        # 放開按鍵時不要再觸發緊急停止
        for k in ("space", "down"):
            pyautogui.keyUp(k)                            # 確保離開時沒有按鍵卡住
        pyautogui.FAILSAFE = True
        sec = time.perf_counter() - t_start
        jumps = sum(1 for e in bot.log if e[1] == "jump")
        ducks = sum(1 for e in bot.log if e[1] == "duck")
        vmax = max((e[3] for e in bot.log if e[1] in ("jump", "duck")), default=bot.speed.v / H)
        print(f"存活 {sec:.1f} 秒｜跳 {jumps} 次、蹲 {ducks} 次｜每秒看 {frames / max(sec, 1e-6):.0f} 張畫面"
              f"｜最高速度 {vmax:.1f} 身高/秒｜量到的延遲 {bot.lag() * 1000:.0f} 毫秒"
              + (f"｜畫面變動、自動重新找恐龍 {relocations} 次" if relocations else ""))
    return bot


def run_dino():
    """一鍵開始：倒數 → 點遊戲、按空白鍵 → 自動找恐龍 → 自動玩到結束"""
    global CAL
    pos = countdown("把滑鼠移到【小恐龍身上】，停住不要動（倒數完會自動開始）", sec=5)
    if pos is None:                           # 倒數時沒移動滑鼠、或滑鼠還在 VS Code 上 → 不開始
        return None
    CAL = start_and_locate(*pos)
    print(f"找到恐龍：身高 H = {CAL['H']} 像素。開始自動玩！")
    return play_dino(CAL)

## A-7 🦖 開始玩！

1. 先用 Chrome **開一個新分頁**打開老師提供的小恐龍網址（例如 `chrome://dino`），畫面停在「還沒開始」或「GAME OVER」都可以。
   程式開始前會自動按 **F5** 重新整理 chrome://dino（慢速模式一旦打開，只有重新整理才會關掉），所以不用擔心「Start slower」。
2. 執行下面這格，**5 秒內切到遊戲視窗，滑鼠放在恐龍身上，不要動，也不要自己點**。
3. 程式會自己點恐龍右邊的空白處、按空白鍵開始、找恐龍、一直玩到結束。撞到後自動停止並印出成績（不會自己重新開始，GAME OVER 畫面上的分數會留著給老師看）。

評分要跑兩次：再執行一次這格就好。

In [ ]:
bot = run_dino()

## A-8 賽後分析

畫出：障礙物速度變化、每次起跳的「剩餘時間」和目標、量到的延遲分布，以及**死前最後幾張截圖**（找出是怎麼死的）。

In [ ]:
# ==========================================================
# A-8 賽後分析：速度變化、每次跳躍的時機、延遲、死前畫面
# ==========================================================
def analyze_dino(bot=None, n_frames=8):
    import matplotlib.pyplot as plt
    bot = bot or LAST_BOT
    if bot is None or not bot.log:
        print("還沒有紀錄，請先執行 A-7 的 run_dino()")
        return
    t0 = bot.log[0][0]
    jumps = [e for e in bot.log if e[1] == "jump"]
    fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
    axes[0].plot([e[0] - t0 for e in bot.log], [e[3] for e in bot.log], ".-")
    axes[0].set_title("障礙物速度（身高/秒）")
    axes[0].set_xlabel("秒")
    axes[1].plot([e[0] - t0 for e in jumps], [e[2] * 1000 for e in jumps], "o", label="起跳時剩餘時間")
    axes[1].plot([e[0] - t0 for e in jumps], [bot.t_jump(e[3] * bot.H) * 1000 for e in jumps], "-", label="目標")
    axes[1].set_title("起跳時機（毫秒）")
    axes[1].legend()
    if bot.lags:
        axes[2].hist(np.array(bot.lags) * 1000, bins=15)
    axes[2].set_title("延遲：按鍵 → 看到起跳（毫秒）")
    plt.tight_layout()
    plt.show()

    frames = list(last_frames)[-n_frames:]
    fig, axes = plt.subplots(len(frames), 1, figsize=(9, 0.9 * len(frames)))
    for ax, (ts, g) in zip(np.atleast_1d(axes), frames):
        ax.imshow(g, cmap="gray", vmin=0, vmax=255)
        ax.set_ylabel(f"{ts:.2f}s", rotation=0, labelpad=25)
        ax.set_xticks([])
        ax.set_yticks([])
    plt.suptitle("死前最後幾張截圖（越下面越新）")
    plt.show()
    print("最後 5 個動作：(事件, 剩幾秒撞到, 速度 身高/秒, 延遲 秒)")
    for e in bot.log[-5:]:
        print(f"  {e[1]:5s}  {e[2]:.3f}  {e[3]:.1f}  {e[4]:.3f}")

In [ ]:
analyze_dino()

## 小恐龍調參指南

| 狀況 | 原因 | 調整 |
|---|---|---|
| 撞到仙人掌前面（太晚跳） | 電腦比較慢、延遲不穩 | `MARGIN` 調大 0.01～0.02 |
| 跳起來落在仙人掌上（太早跳） | 起跳太早 | `MARGIN` 調小 0.01 |
| 快速落地時落在仙人掌上 | 太早按↓ | `DROP_MARGIN` 改成 0.02～0.04 |
| 一直亂跳 | 偵測區有雜訊（廣告、浮水印） | `MIN_DIFF` 調大到 20～30 |
| 每秒看的畫面 < 60，或量到的延遲 > 80 毫秒 | 電腦太忙 | 關掉其他程式、瀏覽器只開遊戲分頁；把 Chrome 視窗縮小一點（街機模式的畫面會跟著變小，瀏覽器比較不吃力） |
| 恐龍跑很慢、畫面有「Start slower」 | 開始前有人點了恐龍 → 進入慢速模式 | 按 F5 重新整理，再執行 A-7（滑鼠放恐龍上就好，不要點） |
| 「找不到恐龍或地面線」 | 遊戲沒有完整出現在螢幕上 | 把遊戲視窗放大、滑鼠放在恐龍附近再試 |

## 小恐龍實驗紀錄（自己測試時填）

| 第幾次 | 改了什麼參數 | 分數 | 存活秒數 | 怎麼死的（看 A-8 的死前截圖） |
|---|---|---|---|---|
| 1 | 預設值 |  |  |  |
| 2 |  |  |  |  |
| 3 |  |  |  |  |

---
# 第 B 部分：寶石方塊（Facebook 版 Bejeweled Blitz）

## 這個遊戲的特性（決定了程式要怎麼寫）

| 特性 | 對程式的影響 |
|---|---|
| 每局只有 1 分鐘 | 比的是「1 分鐘內能有效消除幾次」→ **速度最重要** |
| 寶石還在掉落時，可以交換其他已經停好的寶石 | **加速模式**：不用等整個盤面停下來就下下一步 |
| 連續快速消除會觸發 Blazing Speed（每次消除都像火焰寶石一樣爆開） | 下得越快，分數越高 |
| 消 4 顆變火焰寶石、消 5 顆變超立方體、L/T 形變星星寶石 | 選步時加分；超立方體要主動拿來用 |
| Boost、稀有寶石、加時（Encore）都要金幣，金幣可以花錢買 | 全部不用；程式要在結束畫面出現前停手 |

## 原理

1. **看**：截取整個盤面切成 8×8 格，每格只取**中間 36%** 的區域，取顏色的**中位數**（避開寶石邊框、反光亮點、特效）。
2. **分顏色**：把 RGB 轉成 **Lab 色彩空間**再比較。Lab 是照「人眼感覺」設計的，同一種寶石有亮有暗時，在 Lab 裡還是很接近；不同寶石（例如紅和橘）則分得開。校正時會記下這個遊戲的 7 種寶石顏色，之後每次截圖，同一種寶石都拿到同一個編號。
3. **想**：把所有相鄰交換都模擬一次，每一步都算到底：

| 計分項目 | 為什麼 |
|---|---|
| 消掉的寶石數 × 10 | 消越多分越高 |
| 4 連 +25、5 連 +80、L/T 形 +50 | 會變出特殊寶石（火焰、星星、超立方體），之後能大量消除 |
| 連鎖每多一層 ×1.5 | 消掉後上面掉下來又湊成 3 連，連鎖分數很高 |
| 越靠下面的交換小加分 | 下面消掉，上面整排都會掉落，比較容易引發連鎖 |

4. **做**：用滑鼠交換兩顆寶石。
5. **確認**：等交換的兩格停下來再看顏色——變成別的寶石 = 成功；還是原來的顏色 = 失敗（遊戲沒收到，或消不掉又被換回來），4 秒內不再試同一步。

## 加速模式（邊掉邊下）

每一圈（約 16 毫秒）做這些事：
- 跟 0.1 秒前的截圖比，找出「正在動」的格子；連續 0.08 秒沒動才算靜止。
- **不能用的格子**：正在掉落或消失的；已經 3 連、馬上要被消掉的；剛交換過、等一下會被消掉的那幾行（用模擬預測哪些格子會被消掉）。而且**不能用的格子「上面」的寶石也不能用**（等一下會掉下來）。
- 在剩下的格子裡找最好的一步，馬上下，不用等上一步的動畫結束。

開發測試（自製的 Blitz 型測試版，跟官方一樣可以邊掉邊下，每局 60 秒）：

| 模式 | 1 分鐘內遊戲收到的交換 | 消不掉被換回來 | 分數 |
|---|---|---|---|
| **加速模式** | 108～120 次 | 7～10 次 | 144,250～157,400 |
| 一般模式（每步都等盤面停下來） | 36～38 次 | 0 次 | 80,550～83,250 |

## 三道保險：不亂點、不點到加時

1. **開始前**：畫面上的顏色要對得上 B-5 校正時記下的寶石顏色，而且盤面上沒有現成的 3 連，才算「寶石盤面出現了」。所以在選單、Ready／Go 倒數時不會亂點。
2. **時間**：從第一步成功開始算，59.5 秒自動停（每局 60 秒）。
3. **結束畫面**：盤面七成以上認不出來、又是靜止的，持續 0.8 秒 → 立刻停。開發測試時故意讓程式比遊戲多跑 15 秒，程式都在結束畫面出現時自動停手，沒有點到測試版畫面上的「加時」按鈕。

## 其他自動適應
- **交換方式**：先試拖曳，遊戲完全沒反應就改成「點兩下」。
- **加速模式要不要開**：盤面在動時下的步，遊戲「完全沒反應」（連交換動畫都沒有）才算不接受，連續發生就自動改回一般模式；交換後被換回來只代表那一步算錯，不算。
- **超立方體**：顏色一直在變，所以「連續 0.4 秒認不出是哪種寶石」的格子就當成超立方體候選，跟鄰居中「盤面上最多的那種顏色」交換。試過不是的話，3 秒內不再試。
- **只用靜止 0.3 秒以上的格子學新顏色**：動畫中的過渡色、超立方體的混色都不會被學進去。

In [ ]:
# ==========================================================
# B-1 參數（預設值是針對老師指定的 Facebook 版 Bejeweled Blitz：限時 1 分鐘、寶石掉落時也能繼續下）
# ==========================================================
ROWS, COLS = 8, 8      # 盤面列數、行數
MOVE_MODE  = "auto"    # 交換方式："auto" 自動判斷、"drag" 拖曳、"click" 點兩下
TURBO      = "auto"    # 加速模式：寶石還在掉落時，也在「已經靜止的地方」繼續下（Blitz 允許，分數差很多）
                       # "auto" = 先開著，遊戲不接受就自動關掉；True = 一直開；False = 每步都等盤面停下來
GAME_TIME  = 59.5      # 從「第一步成功」開始算，玩幾秒就停（Blitz 一局 60 秒；提早一點停，避免點到結束畫面的按鈕）
WAIT_START = 180       # 最多等幾秒讓盤面出現（中間要關廣告、按 PLAY、拒絕稀有寶石都來得及；倒數「Ready / Go」時不會亂點）
COLOR_TH   = 20        # 兩個顏色差多少以內算同一種寶石（Lab 色差，越小越嚴格）
PATCH      = 0.36      # 每格只取中間 36% 的區域算顏色（避開邊框、反光、特效）
MOVE_TH    = 12        # 一格的顏色在 0.1 秒內變化超過這個數字，就算「在動」
SETTLE     = 0.10      # 一般模式：整個盤面靜止多久才下（秒）
STILL_TURBO = 0.08     # 加速模式：一格靜止多久就可以用（秒）
COOLDOWN   = 0.5       # 加速模式：剛交換過的兩行，多久之內不要再碰（等一下會消掉、掉落）
LEARN_STILL = 0.3      # 一格要靜止多久，才能拿它的顏色來「學新的寶石種類」（避免學到動畫、超立方體的混色）
MIN_KNOWN  = 0.7       # 盤面靜止時，至少七成的格子認得出是校正時看過的寶石顏色才會下（畫面被蓋住時不亂點）
VERIFY     = 0.9       # 下完一步後，最多等幾秒確認有沒有成功（交換後被換回來 = 失敗）
DRAG_TIME  = 0.10      # 拖曳要花多久（秒）：分成「按下、三段移動、放開」，每段停 DRAG_TIME/4
USE_CUBE   = True      # 會試著使用「超立方體」（消 5 顆變出的特殊寶石，跟任何寶石交換就能消掉同色全部）
BONUS_4, BONUS_5, BONUS_LT = 25, 80, 50   # 消 4 顆、5 顆、L/T 形的額外分數（會變出特殊寶石）
CASCADE    = 0.5       # 連鎖每多一層，分數多乘 0.5 倍
LOW_BONUS  = 0.6       # 越靠下面的交換越容易引發連鎖：每往下一列加 0.6 分

## B-2 寶石方塊的「眼睛」（函式定義）

In [ ]:
# ==========================================================
# B-2 寶石方塊的「眼睛」：截圖 → 每格的顏色 → 分成第幾種寶石
# ==========================================================
def rgb_to_lab(rgb):
    """
    RGB → Lab 顏色空間。Lab 是「照人眼感覺」設計的：兩個顏色在 Lab 裡距離越近，人眼看起來越像。
    用它來比顏色，比直接用 RGB 準很多（例如深紅和淺紅在 RGB 差很多，在 Lab 只差亮度）。
    """
    c = np.asarray(rgb, dtype=float) / 255.0
    c = np.where(c > 0.04045, ((c + 0.055) / 1.055) ** 2.4, c / 12.92)
    M = np.array([[0.4124, 0.3576, 0.1805], [0.2126, 0.7152, 0.0722], [0.0193, 0.1192, 0.9505]])
    xyz = c @ M.T / np.array([0.95047, 1.0, 1.08883])
    f = np.where(xyz > 0.008856, np.cbrt(xyz), 7.787 * xyz + 16 / 116)
    L = 116 * f[..., 1] - 16
    a = 500 * (f[..., 0] - f[..., 1])
    b = 200 * (f[..., 1] - f[..., 2])
    return np.stack([L, a, b], axis=-1)


class Board:
    """記住盤面在螢幕上的位置，負責截圖和算出每格的顏色"""
    def __init__(self, tl, br, rows=ROWS, cols=COLS):
        self.rows, self.cols = rows, cols
        self.cw = (br[0] - tl[0]) / (cols - 1)          # 每格寬 = 兩顆角落寶石中心的距離 ÷ (格數-1)
        self.ch = (br[1] - tl[1]) / (rows - 1)          # 每格高
        self.left = tl[0] - self.cw / 2                 # 盤面左上角
        self.top = tl[1] - self.ch / 2
        pw, ph = max(1, int(self.cw * PATCH / 2)), max(1, int(self.ch * PATCH / 2))
        cx = (np.arange(cols) + 0.5) * self.cw          # 每格中心在截圖裡的 x
        cy = (np.arange(rows) + 0.5) * self.ch          # 每格中心在截圖裡的 y
        # 先算好每格要取哪些像素（之後每次截圖直接用，很快）
        self.ys = (cy[:, None] + np.arange(-ph, ph + 1)[None, :]).astype(int)   # (rows, patch)
        self.xs = (cx[:, None] + np.arange(-pw, pw + 1)[None, :]).astype(int)   # (cols, patch)

    def colors(self):
        """截一張盤面的圖，回傳每格的 Lab 顏色，形狀 (rows, cols, 3)"""
        img = grab(self.left, self.top, self.cw * self.cols, self.ch * self.rows).astype(float)
        h, w = img.shape[:2]
        ys, xs = np.clip(self.ys, 0, h - 1), np.clip(self.xs, 0, w - 1)
        patches = img[ys[:, None, :, None], xs[None, :, None, :]]              # (rows, cols, py, px, 3)
        med = np.median(patches.reshape(self.rows, self.cols, -1, 3), axis=2)  # 用中位數，不怕反光亮點
        return rgb_to_lab(med)

    def center(self, r, c):
        """第 r 列第 c 行那顆寶石，在螢幕上的中心座標"""
        return self.left + (c + 0.5) * self.cw, self.top + (r + 0.5) * self.ch


def find_board_in(img, hx, hy, rows=8, cols=8):
    """
    在截圖 img 裡自動找出 8×8 寶石盤面（(hx, hy) 是盤面上任一點）。
    步驟：① 每個像素判斷「像不像寶石」（亮、或顏色鮮豔）；盤面底色是暗的
          ② 用自相關（圖和「平移後的自己」比對）找出格子大小：寶石每隔一格重複出現
          ③ 試所有格子位置，選「格子中心都是寶石、格子交界都是暗色」分數最高的 8×8
    回傳 (左上寶石中心, 右下寶石中心, 分數)；找不到回傳 None
    """
    lab = rgb_to_lab(img.astype(float))
    L, C = lab[..., 0], np.hypot(lab[..., 1], lab[..., 2])
    G = ((L > 45) & ((C > 28) | (L > 72))).astype(float)
    H, W = G.shape
    # ② 格子大小
    R = 160
    sub = G[max(0, int(hy) - R):int(hy) + R, max(0, int(hx) - R):int(hx) + R]
    sub = sub - sub.mean()
    F = np.fft.rfft2(sub, s=(2 * sub.shape[0], 2 * sub.shape[1]))
    ac = np.fft.irfft2(F * np.conj(F))
    def first_peak(a):
        lo = 12
        seg = a[lo:150]
        if seg.size < 3:
            return None
        mx = seg.max()
        for i in range(1, len(seg) - 1):
            if seg[i] >= seg[i - 1] and seg[i] >= seg[i + 1] and seg[i] >= 0.6 * mx:
                return lo + i
        return None
    px, py = first_peak(ac[0]), first_peak(ac[:, 0])
    if not px or not py:
        return None
    s0 = (px + py) / 2
    # 積分圖：之後「一小塊的平均」只要查 4 個數字
    I = np.zeros((H + 1, W + 1))
    I[1:, 1:] = G.cumsum(0).cumsum(1)
    def box(xs, ys, r):
        x0 = np.clip(np.round(xs - r).astype(int), 0, W); x1 = np.clip(np.round(xs + r + 1).astype(int), 0, W)
        y0 = np.clip(np.round(ys - r).astype(int), 0, H); y1 = np.clip(np.round(ys + r + 1).astype(int), 0, H)
        area = np.maximum((y1[:, None] - y0[:, None]) * (x1[None, :] - x0[None, :]), 1)
        return (I[y1[:, None], x1[None, :]] - I[y0[:, None], x1[None, :]] - I[y1[:, None], x0[None, :]] + I[y0[:, None], x0[None, :]]) / area
    def wsum(a, h, w):                      # 所有 h×w 視窗的總和
        c = np.zeros((a.shape[0] + 1, a.shape[1] + 1)); c[1:, 1:] = a.cumsum(0).cumsum(1)
        return c[h:, w:] - c[:-h, w:] - c[h:, :-w] + c[:-h, :-w]
    best = None
    ix, iy = np.arange(-cols, cols + 1), np.arange(-rows, rows + 1)
    for s in np.arange(s0 - 2, s0 + 2.01, 0.5):
        r = max(1.0, s * 0.12)
        for fx in np.arange(0, s, 1.0):
            cxs = hx - fx + ix * s
            for fy in np.arange(0, s, 1.0):
                cys = hy - fy + iy * s
                cen = box(cxs, cys, r)
                cor = box(np.r_[cxs - s / 2, cxs[-1] + s / 2], np.r_[cys - s / 2, cys[-1] + s / 2], r)
                sc = wsum(cen, rows, cols) / (rows * cols) - wsum(cor, rows + 1, cols + 1) / ((rows + 1) * (cols + 1))
                a, b = np.unravel_index(np.argmax(sc), sc.shape)
                # 滑鼠要在選到的盤面裡面
                if not (cxs[b] - s / 2 <= hx <= cxs[b + cols - 1] + s / 2 and cys[a] - s / 2 <= hy <= cys[a + rows - 1] + s / 2):
                    continue
                if best is None or sc[a, b] > best[0]:
                    best = (sc[a, b], cxs[b], cys[a], cxs[b + cols - 1], cys[a + rows - 1])
    if best is None:
        return None
    sc, ax, ay, bx, by = best
    return (ax, ay), (bx, by), float(sc)


def auto_board(hint_x, hint_y, quiet=False):
    """滑鼠放在盤面上 → 自動找出盤面位置，回傳 Board；找不到回傳 None"""
    R = 420
    sw, sh = pyautogui.size()
    left, top = int(max(0, hint_x - R)), int(max(0, hint_y - R))
    img = grab(left, top, min(2 * R, sw - left), min(2 * R, sh - top))
    found = find_board_in(img, hint_x - left, hint_y - top)
    if found is None or found[2] < 0.5:            # 分數太低 = 沒找到像寶石盤面的東西
        return None
    (ax, ay), (bx, by), score = found
    if not quiet:
        print(f"自動找到盤面（可信度 {score:.2f}）")
    return Board((left + ax, top + ay), (left + bx, top + by))


class Palette:
    """
    記住「每一種寶石的代表色」。第一次看到的顏色就新增一種；之後每格找最像的那一種。
    因為代表色會一直記著，同一種寶石每次都會拿到同一個編號，前後兩張圖可以直接比較。
    """
    def __init__(self, th=COLOR_TH, max_types=14):
        self.th, self.max_types = th, max_types
        self.refs = []                                  # 每種寶石的代表色（Lab）

    def classify(self, lab, still=None):
        """
        lab：每格的顏色；still：哪些格子是靜止的（None = 全部都是靜止的）
        只有「靜止的格子」可以新增或微調代表色；正在動的格子顏色是兩顆寶石混在一起的過渡色，不能學。
        """
        rows, cols, _ = lab.shape
        labels = np.full((rows, cols), -1, dtype=int)
        for r in range(rows):
            for c in range(cols):
                col = lab[r, c]
                learn = still is None or still[r, c]
                if self.refs:
                    d = np.linalg.norm(np.array(self.refs) - col, axis=1)
                    i = int(np.argmin(d))
                    if d[i] < self.th:
                        labels[r, c] = i
                        if learn:
                            self.refs[i] = 0.98 * self.refs[i] + 0.02 * col   # 代表色慢慢微調
                        continue
                if learn and len(self.refs) < self.max_types:
                    self.refs.append(col.copy())       # 沒看過的顏色 → 新的一種寶石
                    labels[r, c] = len(self.refs) - 1
                # 不像任何一種（爆炸特效、正在移動）→ 維持 -1，代表「不確定」
        return labels

## B-3 寶石方塊的「大腦」（函式定義）

`find_matches` 用 numpy 一次比對整個盤面：「每一格跟右邊那格一樣」且「右邊那格跟再右邊一樣」= 3 連。不用一格一格跑迴圈，所以模擬 112 種交換＋連鎖只要十幾毫秒。

In [ ]:
# ==========================================================
# B-3 寶石方塊的「大腦」：模擬每一種交換，算分數，挑最好的
# ==========================================================
def find_matches(b):
    """
    找出會被消掉的格子（橫的或直的連續 3 顆以上一樣）。用 numpy 一次比對整個盤面，很快。
    b：盤面，每格是寶石編號；負數 = 不確定（永遠不會被當成相同）
    回傳：(會消掉的格子 True/False, 加分)
    """
    ok = b >= 0
    eh = (b[:, :-1] == b[:, 1:]) & ok[:, :-1]          # 每一格跟右邊那格一樣嗎
    ev = (b[:-1, :] == b[1:, :]) & ok[:-1, :]          # 每一格跟下面那格一樣嗎
    h3 = eh[:, :-1] & eh[:, 1:]                        # 從這格開始往右 3 顆一樣
    v3 = ev[:-1, :] & ev[1:, :]                        # 從這格開始往下 3 顆一樣
    hh = np.zeros(b.shape, bool)
    vv = np.zeros(b.shape, bool)
    for k in range(3):                                 # 把 3 連的 3 格都標起來
        hh[:, k:b.shape[1] - 2 + k] |= h3
        vv[k:b.shape[0] - 2 + k, :] |= v3
    bonus = 0
    h4 = h3[:, :-1] & h3[:, 1:]                        # 4 連
    v4 = v3[:-1, :] & v3[1:, :]
    bonus += BONUS_4 * (h4.sum() + v4.sum())
    bonus += BONUS_5 * ((h4[:, :-1] & h4[:, 1:]).sum() + (v4[:-1, :] & v4[1:, :]).sum())   # 5 連
    bonus += BONUS_LT * (hh & vv).sum()                # 同一格同時在橫的和直的裡 = L 形或 T 形
    return hh | vv, bonus


def apply_gravity(b, hit):
    """消掉的格子上面的寶石往下掉；最上面空出來的位置填「不確定」（每格一個不同的負數）"""
    rows, cols = b.shape
    new = -1 - np.arange(rows * cols).reshape(rows, cols)
    for c in range(cols):
        keep = b[~hit[:, c], c]
        if keep.size:
            new[rows - keep.size:, c] = keep
    return new


def score_swap(b, a, c):
    """模擬交換 a、c 兩格，回傳分數（不能消就回傳 None），會一路模擬連鎖"""
    bd = b.copy()
    bd[a], bd[c] = bd[c], bd[a]
    total, depth = 0.0, 0
    while depth < 6:
        hit, bonus = find_matches(bd)
        n = int(hit.sum())
        if n == 0:
            break
        total += (n * 10 + bonus) * (1 + CASCADE * depth)
        bd = apply_gravity(bd, hit)
        depth += 1
    if depth == 0:
        return None                                   # 交換後消不掉 → 遊戲會換回來，不能下
    return total + LOW_BONUS * max(a[0], c[0])


def all_moves(b, frozen=None):
    """把所有相鄰交換都試一次，回傳 [(分數, a, c), ...]，分數高的在前面"""
    rows, cols = b.shape
    moves = []
    if frozen is not None:                             # 還在動的格子 → 當作「不確定」
        b = np.where(frozen, -1 - np.arange(b.size).reshape(b.shape), b)
    for r in range(rows):
        for c in range(cols):
            for dr, dc in ((0, 1), (1, 0)):
                r2, c2 = r + dr, c + dc
                if r2 >= rows or c2 >= cols or b[r, c] < 0 or b[r2, c2] < 0 or b[r, c] == b[r2, c2]:
                    continue
                s = score_swap(b, (r, c), (r2, c2))
                if s is not None:
                    moves.append((s, (r, c), (r2, c2)))
    moves.sort(key=lambda m: -m[0])
    return moves

## B-4 動手、主迴圈、校正、檢查（函式定義）

In [ ]:
# ==========================================================
# B-4 動手 + 主迴圈 + 校正 + 檢查
# ==========================================================
def do_swap(board, a, c, mode):
    """用滑鼠交換兩顆寶石：drag = 按住拖過去；click = 點第一顆再點第二顆"""
    x1, y1 = board.center(*a)
    x2, y2 = board.center(*c)
    # 注意：pyautogui.moveTo 的 duration 小於 0.1 秒時會「瞬間跳過去」，真正的遊戲常常收不到拖曳。
    # 所以這裡自己分段移動，每段之間停一下（至少 1 格畫面 ≈ 17 毫秒），讓遊戲每一步都看得到。
    step = DRAG_TIME / 4
    if mode == "drag":
        pyautogui.moveTo(x1, y1)
        time.sleep(step)                                   # 先停在寶石上
        pyautogui.mouseDown()
        time.sleep(step)
        for f in (0.35, 0.7, 1.0):                         # 分三段拖過去
            pyautogui.moveTo(x1 + (x2 - x1) * f, y1 + (y2 - y1) * f)
            time.sleep(step)
        pyautogui.mouseUp()
    else:
        pyautogui.click(x1, y1)
        time.sleep(0.06)
        pyautogui.click(x2, y2)
    pyautogui.moveTo(board.left - 0.6 * board.cw, board.top + board.ch * board.rows / 2)   # 滑鼠移開，不擋住盤面


def looks_like_board(labels):
    """
    盤面看起來像不像寶石盤？（還在選單、「Ready / Go」倒數時，不要亂點）
    條件：5～10 種顏色、沒有哪一種佔超過 40%（選單畫面常常一大片同色）、看不懂的格子很少。
    """
    known = labels[labels >= 0]
    if known.size < labels.size - 6:
        return False
    counts = np.bincount(known)
    kinds = int((counts >= 3).sum())
    return 5 <= kinds <= 10 and counts.max() <= 0.4 * labels.size


def board_check(board, lab):
    """
    這張截圖是不是「真的寶石盤面」？回傳 (是不是, 認得出來的格子比例)
      ‧ 每格的顏色要對得上校正時記下的寶石顏色（選單、廣告、結束畫面都對不上）
      ‧ 盤面上不能已經有 3 連（停好的寶石盤面一定沒有現成的 3 連；選單常常一大片同色）
      ‧ 至少有一步可以下
    """
    refs = getattr(board, "refs0", None)
    if refs is None:                                   # 沒有校正顏色 → 用比較寬鬆的判斷
        labels = Palette().classify(lab)
        return looks_like_board(labels) and bool(all_moves(labels)), 1.0
    d = np.linalg.norm(lab[:, :, None, :] - refs[None, None, :, :], axis=3)
    known = d.min(axis=2) < COLOR_TH
    labels = np.where(known, d.argmin(axis=2), -1 - np.arange(known.size).reshape(known.shape))
    ok = known.mean() >= 0.75 and not find_matches(labels)[0].any() and bool(all_moves(labels))   # 開場的「GO」字會擋住幾格，所以 75% 就算
    return ok, float(known.mean())


def wait_for_board(board, timeout=WAIT_START):
    """
    等寶石盤面出現（還在選單、「Ready / Go」倒數時先不要亂點）。
    等超過 3 秒還沒看到 → 自動在原來位置附近重新找盤面（換電腦、視窗移動、網頁捲動都救得回來），
    找到的盤面如果「顏色種類對、沒有現成 3 連、有步可走」，就用它重新記下寶石顏色。
    每 3 秒印一次進度，讓你知道程式在做什麼。
    """
    t0 = time.perf_counter()
    last_fix = last_print = t0
    while time.perf_counter() - t0 < timeout:
        lab = board.colors()
        ok, frac = board_check(board, lab)
        if ok:
            return True
        now = time.perf_counter()
        if now - last_print > 3:
            last_print = now
            print(f"  還在等盤面出現…（{now - t0:.0f} 秒，認得出的格子 {frac:.0%}）")
        if now - last_fix > 3 and frac < 0.3:              # 幾乎完全認不出來 = 盤面真的跑掉了，才重新找（找一次要好幾秒）
            last_fix = now
            cx, cy = board.center(3.5, 3.5)                    # 原本盤面的正中間
            nb = auto_board(cx, cy, quiet=True)
            if nb is not None:
                lab2 = nb.colors()
                pal = Palette()
                labels = pal.classify(lab2)
                if looks_like_board(labels) and not find_matches(labels)[0].any() and all_moves(labels):
                    refs = getattr(board, "refs0", None)
                    board.__dict__.update(nb.__dict__)         # 換成新找到的位置
                    board.refs0 = np.array(pal.refs)           # 用現在的盤面重新記寶石顏色
                    print("  自動重新找到盤面，開始下！")
                    return True
        time.sleep(0.1)
    return False


def cube_moves(labels, maybe):
    """
    超立方體（消 5 顆變出的特殊寶石）：跟任何寶石交換，就會消掉那個顏色的全部寶石。
    它的顏色一直在變，所以程式「認不出是哪一種寶石」。maybe = 連續 0.4 秒以上都認不出來的格子。
    跟鄰居裡「盤面上最多的那種顏色」交換，一次消最多。回傳 [(預估分數, 超立方體位置, 鄰居位置), ...]
    """
    counts = np.bincount(labels[labels >= 0], minlength=1)
    moves = []
    for r, c in zip(*np.nonzero(maybe)):
        best = None
        for dr, dc in ((0, 1), (1, 0), (0, -1), (-1, 0)):
            r2, c2 = r + dr, c + dc
            if 0 <= r2 < labels.shape[0] and 0 <= c2 < labels.shape[1] and labels[r2, c2] >= 0:
                n = counts[labels[r2, c2]]
                if best is None or n > best[0]:
                    best = (n, (int(r2), int(c2)))
        if best and best[0] >= 5:
            moves.append((best[0] * 15 + 100, (int(r), int(c)), best[1]))
    return moves


GEM_LOG = []          # 每一步的紀錄：(第幾秒, 預估分數, a, c, 成功與否, 下的時候盤面在不在動)


def play_gem(board, game_time=GAME_TIME):
    """
    自動玩。每一圈：截圖 → 找出「在動」和「靜止」的格子 → 檢查之前的步有沒有成功 → 找最好的一步 → 交換。
    加速模式下不必等上一步的動畫結束，只要避開正在動的格子，就能馬上在別的地方下下一步。
    """
    if not wait_for_board(board):
        print(f"等了 {WAIT_START} 秒都沒看到寶石盤面：請確認遊戲已經開始、校正位置正確")
        return None
    from collections import deque
    pal = Palette()
    refs0 = getattr(board, "refs0", None)
    if refs0 is not None:
        pal.refs = [r.copy() for r in refs0]          # 一開始就用校正時記下的寶石顏色
    mode = MOVE_MODE if MOVE_MODE != "auto" else "drag"
    mode_tested = MOVE_MODE != "auto"
    turbo = TURBO in (True, "auto")
    turbo_ok = turbo_fail = 0
    R, C = board.rows, board.cols
    hist = deque()                          # 最近 0.1 秒的截圖：(時間, 顏色)
    last_change = np.full((R, C), -9.0)     # 每一格「最後一次在動」的時間
    busy_start = np.full((R, C), np.inf)    # 每一格「這一次連續在動」是從什麼時候開始
    cool_until = np.zeros((R, C))           # 每一格冷卻到什麼時候（剛交換過，等一下會消掉、掉落）
    pending = []                            # 還沒確認的步：(下的時間, a, c, key, 當時盤面在動嗎, 預估分數, 交換前兩格的顏色)
    bad = {}                                # 失敗過的步 → 失敗的時間（4 秒內不再試）
    busy_since = 0.0                        # 最後一次「超過 2 格在動」的時間
    unknown_since = np.full((R, C), np.inf) # 每一格「連續認不出是哪種寶石」從什麼時候開始
    covered_since = None                    # 盤面「大部分認不出來」從什麼時候開始（被結束畫面蓋住？）
    not_cube = {}                           # 試過「當超立方體交換」卻失敗的格子 → 失敗時間
    t0 = time.perf_counter()                # 看到盤面的時間（遊戲差不多從這時開始倒數 1 分鐘）
    first_ok = None                         # 第一步成功的時間
    moves_done = fails = noreact_in_row = 0
    GEM_LOG.clear()
    try:
        while True:
            now = time.perf_counter()
            # 計時起點：第一步成功的時間，但最晚不超過「看到盤面後 2.5 秒」（開場「GO」之前下的步遊戲不收）
            start = None if first_ok is None else min(first_ok, t0 + 2.5)
            if start is not None and now - start > game_time:
                print(f"{game_time:g} 秒到了，自動停止（避免點到結束畫面的按鈕）")
                break
            if first_ok is None and now - t0 > WAIT_START:
                print("一直沒有成功下出一步，自動停止（遊戲開始了嗎？）")
                break
            lab = board.colors()

            # ---- 1. 哪些格子在動？跟 0.1 秒內最舊的那張截圖比（慢慢淡出的動畫也抓得到）----
            hist.append((now, lab))
            while now - hist[0][0] > 0.1:
                hist.popleft()
            moving = np.linalg.norm(lab - hist[0][1], axis=2) > MOVE_TH
            last_change[moving] = now
            busy_start = np.where(moving, np.minimum(busy_start, now), np.inf)
            flicker = moving & (now - busy_start > 0.6)          # 連續動了 0.6 秒以上 = 原地閃爍的特殊寶石
            still = now - last_change > (STILL_TURBO if turbo else SETTLE)
            if (~still & ~flicker).sum() > 2:
                busy_since = now
            labels = pal.classify(lab, still=now - last_change > LEARN_STILL)   # 只用「很穩定」的格子學新顏色
            unknown_since = np.where(labels < 0, np.minimum(unknown_since, now), np.inf)
            # 認得出來的格子比例：太低代表盤面被蓋住（Time Up、結束畫面、彈出視窗）→ 這一圈不下
            # 用「隨時更新的寶石顏色」來算：Blazing Speed 時背景變成火焰色、寶石顏色也跟著變，
            # 如果只跟開局記下的顏色比，會誤以為盤面被蓋住而不下（錄影裡最後 5 秒就是這樣停掉的）
            known_frac = float((labels >= 0).mean())
            known0 = 1.0                                           # 跟開局顏色比（只用來判斷「結算畫面蓋住」）
            if refs0 is not None:
                known0 = float((np.linalg.norm(lab[:, :, None, :] - refs0[None, None], axis=3).min(axis=2)
                                < COLOR_TH).mean())
            moving_frac = float((~still).mean())
            # 「認不出來」又「沒在動」= 被靜止的畫面蓋住（彈出視窗、結束畫面）；連鎖消除時雖然認不出來，但是在動
            covered = known_frac < MIN_KNOWN and moving_frac < 0.25
            # 遊戲開始後，盤面七成以上認不出來、又沒在動，持續 0.8 秒 → 被「Time Up / 結束畫面」蓋住了 → 立刻停手
            if first_ok is not None:
                if known0 < 0.15 and moving_frac < 0.25:
                    # 幾乎完全認不出來、又不動，連續 3 秒 → 真的被結算畫面蓋住了才停（特效再久也不會到 3 秒）
                    covered_since = covered_since or now
                    if now - covered_since > 3.0:
                        print("盤面被蓋住了（時間到或結束畫面）→ 自動停止，不會點到結束畫面的按鈕")
                        break
                else:
                    covered_since = None

            # ---- 2. 檢查之前下的步有沒有成功 ----
            # 等那兩格停下來後比顏色：變成別的寶石 = 成功（被消掉、新的掉進來）；
            # 還是原來的顏色 = 失敗（遊戲沒收到，或交換後消不掉又被換回來）
            for item in list(pending):
                tm, a, c, key, was_busy, sc, before = item
                settled = still[a] and still[c] and now - tm > 0.3
                if not (settled or now - tm > VERIFY):
                    continue
                pending.remove(item)
                after = lab[[a[0], c[0]], [a[1], c[1]]]
                ok = bool((np.linalg.norm(after - before, axis=1) > MOVE_TH).any())
                reacted = ok or last_change[a] > tm + 0.03 or last_change[c] > tm + 0.03   # 有沒有交換動畫
                GEM_LOG.append((round(tm - t0, 2), round(sc), a, c, ok, was_busy))
                if reacted:
                    mode_tested = True                    # 遊戲有反應 → 這種交換方式是對的
                    noreact_in_row = 0
                else:
                    noreact_in_row += 1
                if ok:
                    turbo_ok += was_busy
                    if first_ok is None:
                        first_ok = tm
                        bad.clear()                               # 開始前（Ready/Go）失敗的步不算
                else:
                    fails += 1
                    bad[key] = now
                    if key[2] < 0:
                        not_cube[a] = now                 # 這格不是超立方體（或還不能用），3 秒內不再試
                    if not reacted and not mode_tested:
                        mode = "click" if mode == "drag" else "drag"   # 完全沒反應 → 換另一種交換方式試試看
                    elif not reacted and was_busy and turbo and first_ok is not None:
                        # 遊戲開始後，盤面在動時下、遊戲完全沒反應 → 這個遊戲可能不接受「邊掉邊下」
                        turbo_fail += 1
                        if TURBO == "auto" and turbo_fail >= 2 and turbo_fail > turbo_ok:
                            turbo = False
                            print("（這個遊戲在寶石掉落時不接受操作，自動改成「等盤面停下來再下」）")
            if first_ok is not None and noreact_in_row >= 6:
                # 不再直接停止（中途停掉太可惜）：暫停 0.6 秒、重新確認盤面位置，再繼續；1 分鐘時間到才會停
                print("（連續 6 步沒反應 → 暫停一下、重新確認盤面再繼續）")
                time.sleep(0.6)
                if board_check(board, board.colors())[1] < 0.3:
                    wait_for_board(board, timeout=5)
                noreact_in_row = 0

            if covered:                                      # 盤面被蓋住 → 先不要點
                time.sleep(0.005)
                continue

            # ---- 3. 哪些格子不能用？----
            if turbo:
                falling = ~still & ~flicker                      # 正在掉落、消失的格子
                frozen = falling | find_matches(labels)[0] | (cool_until > now)
                # 不能用的格子「上面」的寶石等一下也會掉下來 → 一起不能用
                frozen = np.maximum.accumulate(frozen[::-1], axis=0)[::-1] | flicker
                board_busy = bool(falling.any() or pending)
            else:
                # 一般模式：上一步確認完、整個盤面停下來才下（容許 2 格一直閃爍）
                if pending or (~still).sum() > 2 or ((~still).any() and now - busy_since < 3 * SETTLE):
                    time.sleep(0.005)
                    continue
                frozen = ~still
                board_busy = False

            # ---- 4. 找最好的一步（超立方體優先考慮）----
            cand = all_moves(labels, frozen)
            if USE_CUBE:
                maybe = (labels < 0) & (now - unknown_since > 0.4) & ~(cool_until > now)
                for (r, c), tf in not_cube.items():
                    if now - tf < 3:
                        maybe[r, c] = False
                cand = sorted(cand + cube_moves(labels, maybe), key=lambda m: -m[0])
            cand = [m for m in cand if now - bad.get((m[1], m[2], labels[m[1]], labels[m[2]]), -99) > 4]
            if not cand:
                time.sleep(0.005)
                continue
            score, a, c = cand[0]

            # ---- 5. 動手 ----
            do_swap(board, a, c, mode)
            moves_done += 1
            tm = time.perf_counter()
            # 預測哪些格子會被消掉：那幾行（從最上面到被消掉的最低那格）等一下都會掉落 → 冷卻
            bd = labels.copy()
            bd[a], bd[c] = bd[c], bd[a]
            hit = find_matches(bd)[0]
            hit[a] = hit[c] = True
            if labels[a] < 0 or labels[c] < 0:            # 超立方體：會消掉一整種顏色，整個盤面都會動
                hit[:, :] = True
            for col in np.flatnonzero(hit.any(axis=0)):
                low = np.flatnonzero(hit[:, col]).max()
                cool_until[:low + 1, col] = tm + COOLDOWN
            before = lab[[a[0], c[0]], [a[1], c[1]]].copy()
            pending.append((tm, a, c, (a, c, labels[a], labels[c]), board_busy, score, before))
    except KeyboardInterrupt:
        print("手動停止")
    except pyautogui.FailSafeException:
        print("滑鼠移到螢幕左上角 → 緊急停止")
    finally:
        pyautogui.FAILSAFE = False
        pyautogui.mouseUp()                           # 確保滑鼠按鍵沒有卡住
        pyautogui.FAILSAFE = True
    played = time.perf_counter() - (first_ok or t0)
    ok_moves = sum(1 for e in GEM_LOG if e[4])
    print(f"共下了 {moves_done} 步（成功 {ok_moves} 步），平均每秒 {ok_moves / max(played, 1e-6):.1f} 步，"
          f"交換方式：{mode}，加速模式：{'開' if turbo else '關'}，辨識出 {len(pal.refs)} 種顏色")
    return pal


BOARD = None


def calibrate_gem():
    """
    校正（要在「遊戲進行中、看得到整個盤面」時做）：滑鼠移到左上角、右下角兩顆寶石的正中間。
    同時記下這個遊戲的寶石顏色（之後用來判斷「現在畫面上是不是真的寶石盤面」）。
    """
    global BOARD
    pos = countdown("把滑鼠移到【寶石盤面上】（盤面中間任何一顆寶石都可以），停住不要動", sec=10)
    if pos is None:
        return None
    BOARD = auto_board(*pos)                   # 自動找盤面：不用再對準兩個角
    if BOARD is None:                          # 自動找不到 → 改用手動：對準兩個角
        print("自動找不到盤面，改用手動校正")
        tl = countdown("把滑鼠移到【左上角第一顆寶石的正中間】", sec=10)
        if tl is None:
            return None
        br = countdown("把滑鼠移到【右下角最後一顆寶石的正中間】", sec=10)
        if br is None:
            return None
        BOARD = Board(tl, br)
    pal = Palette()
    pal.classify(BOARD.colors())
    BOARD.refs0 = np.array(pal.refs)
    print(f"每格大小約 {BOARD.cw:.1f} × {BOARD.ch:.1f} 像素，記下 {len(pal.refs)} 種寶石顏色")
    if not 5 <= len(pal.refs) <= 9:
        print("⚠️ 顏色種類數不太對（寶石方塊通常是 7 種）：請確認校正時盤面沒有被擋住，或調整 COLOR_TH 後重新校正")
    return BOARD


def show_board(board=None):
    """把程式「看到」的盤面畫出來：每格寫上辨識出的寶石編號，相同數字 = 同一種寶石"""
    import matplotlib.pyplot as plt
    board = board or BOARD
    if board is None:
        print("還沒有校正：請在遊戲進行中執行 calibrate_gem()（倒數時把滑鼠移到寶石上）")
        return
    img = grab(board.left, board.top, board.cw * board.cols, board.ch * board.rows)
    labels = Palette().classify(board.colors())
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.imshow(img)
    for r in range(board.rows):
        for c in range(board.cols):
            ax.text((c + 0.5) * board.cw, (r + 0.5) * board.ch, str(labels[r, c]), color="white",
                    ha="center", va="center", fontsize=14, fontweight="bold",
                    bbox=dict(facecolor="black", alpha=0.5, pad=1))
    moves = all_moves(labels)
    if moves:
        _, a, c = moves[0]
        (y1, x1), (y2, x2) = a, c
        ax.annotate("", xy=((x2 + 0.5) * board.cw, (y2 + 0.5) * board.ch),
                    xytext=((x1 + 0.5) * board.cw, (y1 + 0.5) * board.ch),
                    arrowprops=dict(color="yellow", lw=3, arrowstyle="<->"))
    ax.set_title(f"辨識出 {labels.max() + 1} 種寶石｜黃色箭頭 = 程式認為最好的一步")
    ax.axis("off")
    plt.show()
    print(labels)
    print(f"可以下的步數：{len(moves)}；最好的一步：{moves[0][1]} ↔ {moves[0][2]}（預估 {moves[0][0]:.0f} 分）"
          if moves else "找不到可以下的步：請檢查顏色辨識（COLOR_TH）或校正位置")


def run_gem(recalibrate=False):
    """
    一鍵開始：執行後切到遊戲視窗、按下開始（Play）。
    程式會等「真的寶石盤面」出現才開始下（Ready / Go 倒數時不會亂點），第一步成功後 59.5 秒自動停止。
    """
    if BOARD is None or recalibrate:
        if calibrate_gem() is None:
            return None
    if countdown("切到遊戲視窗、按下開始（Play）；程式會等寶石盤面出現才開始下", sec=10) is None:
        return None
    print("（要停：滑鼠甩到螢幕左上角，或按 Jupyter 的 ■）")
    return play_gem(BOARD)

## B-5 校正 + 檢查顏色辨識（正式玩之前一定要做）

1. 用 Chrome 打開老師給的網址並登入 Facebook，**開一局練習**（這局分數不重要）。
2. 遊戲進行中，執行下面這格：10 秒內把滑鼠移到**寶石盤面上**（中間任何一顆寶石都可以），停住不動，程式會**自動找出整個盤面**（找不到才會改成請你對準左上角、右下角兩顆寶石）。
3. 檢查畫出來的圖：**同一種寶石要是同一個數字**，種類數大約是 7。黃色箭頭是程式認為最好的一步。

辨識不對時：同色被分成不同數字 → `COLOR_TH` 調大；不同色被當成同一個數字 → `COLOR_TH` 調小。改完要重新執行 B-1 和這一格。
只要瀏覽器視窗不移動、網頁不捲動，校正一次就可以一直用。

In [ ]:
calibrate_gem()
show_board()

## B-6 💎 開始玩！

1. 執行下面這格，**10 秒內切到遊戲視窗**。
2. 在遊戲裡按 PLAY（跳出稀有寶石就按 NO THANKS），確認**沒有選任何 Boost**（Boost 欄位是空的）。
3. 手不要碰滑鼠。程式會等寶石盤面出現才開始下（Ready／Go 倒數時不會亂點），1 分鐘到或出現結束畫面就自動停。

評分要跑兩次：再執行一次這格就好。
- 程式最多等 3 分鐘讓盤面出現，按 ▶ 之後慢慢關廣告、按 PLAY、拒絕稀有寶石（NO THANKS）都來得及。
- 上一局結束後如果卡在分數畫面、看不到按鈕，按 **F5** 重新整理回到主選單再開始；盤面位置變了也沒關係，程式會自己重新找。
- 遊戲視窗移動過、程式一直找不到盤面的話，改成 `run_gem(recalibrate=True)`。

In [ ]:
run_gem()

## B-7 賽後分析

In [ ]:
# ==========================================================
# B-7 賽後分析：每一步的時間、預估分數、成功與否
# ==========================================================
def analyze_gem():
    import matplotlib.pyplot as plt
    if not GEM_LOG:
        print("還沒有紀錄，請先執行 B-6 的 run_gem()")
        return
    t = np.array([e[0] for e in GEM_LOG])
    sc = np.array([e[1] for e in GEM_LOG])
    ok = np.array([e[4] for e in GEM_LOG])
    fig, axes = plt.subplots(1, 2, figsize=(13, 3.5))
    axes[0].plot(t[ok], np.arange(len(t))[ok] + 1, "g.", label="成功")
    axes[0].plot(t[~ok], np.arange(len(t))[~ok] + 1, "rx", label="沒反應")
    axes[0].set_xlabel("秒")
    axes[0].set_title("第幾步 vs. 時間（越陡 = 下得越快）")
    axes[0].legend()
    axes[1].hist(sc[ok], bins=15)
    axes[1].set_title("每一步的預估分數分布")
    plt.tight_layout()
    plt.show()
    gaps = np.diff(t)
    print(f"共 {len(t)} 步，成功 {ok.sum()} 步；平均每步間隔 {gaps.mean() if gaps.size else 0:.2f} 秒；"
          f"最高預估分數的一步：{sc.max():.0f}")

In [ ]:
analyze_gem()

## 寶石方塊調參指南

| 狀況 | 調整 |
|---|---|
| 同色寶石被分成不同編號 | `COLOR_TH` 調大（例如 25～30），重新校正 |
| 不同色寶石被當成同一種 | `COLOR_TH` 調小（例如 14～16），重新校正 |
| 程式一直不開始下 | 重新做 B-5 校正（瀏覽器視窗可能移動過） |
| 常常「消不掉被換回來」 | `STILL_TURBO` 調大到 0.12（等寶石停穩一點再用） |
| 遊戲不接受「邊掉邊下」 | 程式會自動改回一般模式；也可以直接把 `TURBO` 設成 `False` |
| 時間還沒到就停了 | 看印出來的停止原因；`GAME_TIME` 不要超過 59.5 |

## 寶石方塊實驗紀錄（自己測試時填）

| 第幾次 | 改了什麼參數 | 分數 | 下了幾步 | 觀察 |
|---|---|---|---|---|
| 1 | 預設值 |  |  |  |
| 2 |  |  |  |  |
| 3 |  |  |  |  |

---
# 第 C 部分：自我測試（不用開遊戲）

用「假的資料」檢查兩個大腦算得對不對：寶石方塊的 3 連／4 連／L 形判斷、掉落、選步、「是不是真的寶石盤面」、超立方體；小恐龍在三種速度下的起跳時機、快速落地時機、量速度的正確性。
每一項都會印出 ✅ 或 ❌。改過程式之後跑一次，確定沒有改壞。

In [ ]:
# ==========================================================
# C. 自我測試：不用開遊戲，用「假的畫面資料」檢查兩個大腦算得對不對
# ==========================================================
def selftest():
    passed = 0

    def check(name, cond):
        nonlocal passed
        print(("✅ " if cond else "❌ ") + name)
        passed += bool(cond)
        return cond

    print("【寶石方塊大腦】")
    b = np.array([[0, 0, 0, 1],
                  [1, 2, 3, 4],
                  [2, 3, 4, 5],
                  [3, 4, 5, 6]])
    hit, bonus = find_matches(b)
    check("找到第一列的 3 連（只有那 3 格會消）", hit[0, :3].all() and hit.sum() == 3 and bonus == 0)
    hit, bonus = find_matches(np.array([[2, 2, 2, 2, 5]]))
    check("4 連有額外加分（會變出特殊寶石）", hit.sum() == 4 and bonus == BONUS_4)
    lt = np.array([[1, 5, 6],
                   [1, 7, 8],
                   [1, 1, 1]])
    hit, bonus = find_matches(lt)
    check("L 形（橫 3 + 直 3）有額外加分", hit.sum() == 5 and bonus >= BONUS_LT)
    g = apply_gravity(np.array([[1], [2], [3]]), np.array([[False], [True], [False]]))
    check("消掉中間那格後，上面的寶石往下掉、最上面變成「不確定」", g[2, 0] == 3 and g[1, 0] == 1 and g[0, 0] < 0)
    board = np.array([[0, 1, 2, 3, 4, 5],
                      [1, 2, 3, 4, 5, 0],
                      [6, 6, 0, 6, 6, 1],
                      [2, 3, 6, 5, 1, 2]])
    best = all_moves(board)[0]
    check("有 3 連和 5 連可以選時，選 5 連那一步", {best[1], best[2]} == {(2, 2), (3, 2)})
    check("同顏色交換、消不掉的交換都不會被選", all(score_swap(board, m[1], m[2]) is not None for m in all_moves(board)))

    import types
    rng = np.random.default_rng(7)
    refs = rgb_to_lab(np.array([[230, 40, 40], [250, 150, 40], [240, 230, 60], [60, 220, 100],
                                [60, 150, 250], [190, 90, 250], [240, 240, 240]]))
    true = np.zeros((8, 8), int)
    for r in range(8):                                   # 做一個「沒有現成 3 連」的隨機盤面
        for c in range(8):
            while True:
                t = rng.integers(0, 7)
                if not ((c >= 2 and true[r, c - 1] == t == true[r, c - 2]) or (r >= 2 and true[r - 1, c] == t == true[r - 2, c])):
                    break
            true[r, c] = t
    fake = types.SimpleNamespace(refs0=refs)
    lab = refs[true] + rng.normal(0, 2, (8, 8, 3))
    check("真的寶石盤面 → 判斷為「可以開始下」", board_check(fake, lab)[0])
    check("一整片灰色的選單畫面 → 判斷為「不是寶石盤面」，不會亂點",
          not board_check(fake, np.full((8, 8, 3), 60.0))[0])
    cube = true.copy()
    cube[3, 3] = -1                                      # 認不出來的格子（超立方體候選）
    mv = cube_moves(cube, cube < 0)
    most = max(((r, c) for r, c in ((3, 4), (4, 3), (3, 2), (2, 3))), key=lambda p: (true == true[p]).sum())
    check("超立方體會跟「盤面上最多的那種顏色」的鄰居交換", bool(mv) and mv[0][2] == most)

    print("\n【小恐龍大腦】用假的障礙物，模擬三種速度，看起跳時機和快速落地時機")
    pressed = []
    real = (pyautogui.keyDown, pyautogui.keyUp)
    pyautogui.keyDown = lambda k: pressed.append((k, "按下"))
    pyautogui.keyUp = lambda k: pressed.append((k, "放開"))
    try:
        cal = {"H": 47, "NOSE_X": 94, "DINO_LEFT": 50, "GROUND_Y": 140, "DINO_TOP": 93, "BG": 247, "CONTRAST": 164}
        print("速度(身高/秒) | 起跳時距離(像素) | 起跳時剩幾毫秒撞到 | 目標(毫秒) | 按↓時障礙物已過恐龍尾巴(像素)")
        rows_ok = True
        for speed in (7.66, 12.0, 16.6):                 # 原版：開始 7.66、最高 16.6 身高/秒
            bot = DinoBot(cal)
            bot.lags = [0.05]                            # 假設量到的延遲是 50 毫秒
            v, x, t, w = speed * 47, 6 * 47.0, 0.0, 17   # 一個寬 17 像素的小仙人掌，從 6 倍身高外跑過來
            jump = drop = None
            while x > -200 and t < 3:
                low = (int(x), int(x) + w) if x + w >= 0 and x < 6 * 47 else None
                if low and low[0] < 0:
                    low = (0, low[1])
                legs = 0 if (jump and t - jump[0] > 0.05) else 400       # 起跳 50 毫秒後腳離地
                bot.step(t, low, None, legs)
                if jump is None and bot.state == "AIR":
                    jump = (t, x, (x + GAP * 44) / v, bot.t_jump(v))
                if jump and drop is None and bot.state == "DROP":
                    tail_gap = -(x + w + GAP * 44 + 44)  # 障礙物尾巴在恐龍尾巴後面多少
                    drop = (t, tail_gap)
                t += 1 / 120
                x -= v / 120
            if jump is None or drop is None:
                rows_ok = False
                continue
            print(f"{speed:>12.2f} | {jump[1]:>15.0f} | {jump[2] * 1000:>17.0f} | {jump[3] * 1000:>9.0f} | {drop[1]:>10.0f}")
            rows_ok &= abs(jump[2] - jump[3]) < 0.02 and drop[1] >= -1
        check("三種速度都在目標時機起跳，而且等障礙物過了尾巴才快速落地", rows_ok)
        check("跳躍時有按下、放開空白鍵，快速落地有按↓", ("space", "按下") in pressed and ("down", "按下") in pressed)
    finally:
        pyautogui.keyDown, pyautogui.keyUp = real

    sm = SpeedMeter(100.0)
    for i in range(30):                                 # 每 1/60 秒移動 10 像素 = 600 像素/秒，但同一格畫面被截到 3 次
        for _ in range(3):
            sm.update(300 - 10 * i, i / 60 + _ * 0.002)
    check(f"量速度不會被「同一格畫面截到好幾次」騙到（量到 {sm.v:.0f}，正確 600 像素/秒）", abs(sm.v - 600) < 60)
    print(f"\n通過 {passed} 項測試")


selftest()

---
# 開發過程紀錄

開發時用 **Xvfb 虛擬螢幕 + Chromium 瀏覽器 + 原版 Chrome 恐龍遊戲原始碼**，以及自製的兩個寶石方塊測試版（一個要等盤面停下來才能下、一個跟 Blitz 一樣可以邊掉邊下），讓程式用真的截圖、真的鍵盤滑鼠去玩，邊測邊修。

| 發現的問題 | 原因 | 解法 |
|---|---|---|
| 按下開始後找到的恐龍大小不對 | 新版遊戲開始約 2 秒後會把畫面放大 1.5 倍（街機模式），網頁也會往下捲 21 像素 | 開始後每 0.25 秒量一次，連續兩次一樣才採用 |
| 在 chrome://dino 一開始就找不到恐龍（只拿 42 分） | 街機模式把恐龍放大約 3 倍、移到螢幕左下角，跑出「滑鼠附近」的搜尋範圍 | 附近找不到就搜整個螢幕 |
| chrome://dino 恐龍變很慢、一次都不跳（44 分） | 程式先點恐龍再按空白鍵 → Chrome 以為在用鍵盤導覽，自動打開「Start slower」慢速模式；慢速時該起跳的那一刻，障礙物已經在「緩衝區」裡，眼睛只看到位置 0，算出來永遠「還沒到」 | ① 改點恐龍右邊的空白處，不會進入慢速模式 ② 緩衝區裡用速度推算位置 |
| 高速時偶爾太晚蹲，被飛鳥撞 | 畫面卡頓時速度量偏低，而且蹲下只提早 0.05 秒 | 速度改用直線擬合＋「最近障礙物速度」當下限；蹲下提早到 0.15 秒（蹲早沒關係） |
| 跳過很寬的仙人掌群時，程式以為已經落地、又按了一次跳 | 仙人掌從恐龍腳下經過，被當成「腳又出現了」 | 障礙物尾巴過了恐龍之後，才承認落地 |
| 撞到之後，GAME OVER 畫面被程式按成重新開始 | 撞到後畫面凍結，程式還在推算「障礙物越來越近」而按跳 | 障礙物 0.5 秒完全不動 → 進入 STUCK，不再按任何鍵 |
| 電腦忙的時候延遲忽快忽慢（0.05～0.17 秒），偶爾一次太晚跳 | 用中位數估延遲，碰到慢的那一次就來不及 | 改用第 70 百分位數，高速時再多提早 0.05 秒 |
| 恐龍落地後一直原地亂跳 | 快速落地時按著↓，落地瞬間恐龍是蹲著的，蹲下比站著長，頭伸進偵測區被當成仙人掌 | 偵測區起點改成離嘴巴 0.5 倍「身寬」 |
| 跳太晚撞到兩個連在一起的仙人掌 | 起跳時機算法沒考慮「真正撞到的是腳」 | 改成用遊戲物理推算，並扣掉腳在嘴巴後面的距離 |
| 700 分夜晚模式後突然看不到東西 | 捲動防護把「瀏覽器視窗下緣的黑邊」誤認成地面線 | 地面線必須是「細線」，而且上面要站著一樣大的恐龍才算 |
| 量速度一開始猜錯就修不回來 | 為了排除雜訊，和目前速度差 2 倍以上的量測值都丟掉 | 資料夠多（0.1 秒、5 個位置）時直接相信量到的值 |
| 寶石方塊偶爾在動畫快結束時就出手，遊戲沒收到 | 只跟上一張截圖比，慢慢淡出的動畫每張差一點點，被當成靜止 | 改成跟「0.1 秒內的每一張」比 |
| 試完加速模式後辨識變差 | 動畫中的過渡色被學成新的寶石顏色 | 只用靜止 0.3 秒以上的格子學顏色 |
| 加速模式下有一半的交換消不掉 | 只冷卻了交換的那兩行，但消掉的寶石可能橫跨三行，那些行上面的寶石等一下會掉 | 用模擬預測「哪些格子會被消掉」，把那幾行全部冷卻 |
| 把「消不掉被換回來」也算成功 | 只看交換的格子有沒有動過 | 改成看最後的顏色：被換回原來的顏色 = 失敗 |
| 有一局只下了 2 步就卡住 | 盤面上只剩「超立方體」可以下，程式認不出它 | 連續認不出顏色的格子當成超立方體候選，主動拿來用 |
| 一局玩到一半就自己停了 | 「連續 4 步失敗就停」被正常的失敗誤觸 | 只有「遊戲完全沒反應」連續 6 次才停；結束畫面另外用「盤面被蓋住」判斷 |
| 擔心點到結束畫面的加時按鈕（要花金幣） | 遊戲時間到了程式還在下 | 59.5 秒自動停＋盤面被蓋住就停（測試時故意多跑 15 秒，也沒點到） |

**開發測試成績**
- 小恐龍（原版原始碼）：chrome://dino 一般模式 9 分鐘沒撞到（10,220 分）；慢速模式（Start slower）5 分鐘沒撞到；電腦很忙、每秒只剩約 30 格畫面時也撐了約 6 分鐘（6,471 分）；原版原始碼最長 10 分鐘沒撞到（11,388 分）
- 寶石方塊（自製測試版，60 秒限時）：Blitz 型測試版連續 3 局：144,250／147,850／157,400 分，遊戲收到 108～120 次交換，消不掉被換回來 7～10 次；不能邊掉邊下的測試版：加速模式自動關閉，1 分鐘 60 次交換、0 次無效交換；兩種測試版開局的顏色辨識都和真實盤面 100% 一致

> 真正的分數以老師提供的網址為準。不同網站的畫面、速度、延遲都可能不同，所以程式才設計成「自動校正、自動量速度、自動量延遲」。

# 參考資料
- Chromium 恐龍遊戲原始碼（t-rex-runner）：https://github.com/wayou/t-rex-runner
- Lab 色彩空間：https://zh.wikipedia.org/wiki/CIELAB色彩空間
- pyautogui 文件：https://pyautogui.readthedocs.io/
- mss 文件：https://python-mss.readthedocs.io/